# LeetCode 262 — Trips and Users (Hard, Database) bằng SQL + Python

Đây là bài thuộc nhóm **Database** (cơ sở dữ liệu). Khác với các bài thuật toán, ở đây bạn **không viết `class Solution`**. Trên LeetCode bạn nộp **một câu truy vấn SQL** (chọn ngôn ngữ **MySQL**), hoặc một hàm **Pandas**. Khung code LeetCode cho sẵn:

```sql
# Write your MySQL query statement below
```

```python
import pandas as pd

def trips_and_users(trips: pd.DataFrame, users: pd.DataFrame) -> pd.DataFrame:

```

Bài này được gắn nhãn "Hard" **không phải** vì thuật toán phức tạp. Nó khó vì có **rất nhiều cái bẫy nhỏ**: phải lọc **cả khách lẫn tài xế** bị cấm, phải nhớ **khoảng ngày**, phải chia ra **số thực** chứ không phải số nguyên, phải **làm tròn** đúng chỗ, phải đặt **tên cột có dấu cách**… Chỉ cần sai một chỗ là "Wrong Answer" dù ví dụ trong đề vẫn chạy đúng.

Notebook này giả sử bạn **chưa biết gì về SQL**. Ta dùng `sqlite3`, một cơ sở dữ liệu mini **có sẵn trong Python**, để tạo bảng thật, chạy query thật và nhìn kết quả thật ngay trong notebook.

---

## Sau notebook này, bạn sẽ nắm được

1. **Bảng, hàng, cột, khóa chính, khóa ngoại**: những khái niệm nền của cơ sở dữ liệu quan hệ.
2. **SELECT / WHERE / BETWEEN / ORDER BY**, và vì sao ngày dạng chuỗi `'YYYY-MM-DD'` so sánh được.
3. **JOIN**: ghép hai bảng, **alias** (tên tắt), và **JOIN cùng một bảng hai lần** (một lần cho khách, một lần cho tài xế).
4. **Subquery** với **IN / NOT IN**, kèm cái bẫy `NULL`.
5. **GROUP BY, COUNT, SUM(CASE WHEN …)**, **AVG** trên biểu thức đúng/sai, **ROUND**.
6. **Chia số nguyên hay số thực**: sqlite **khác** MySQL ở đâu, và vì sao ta viết `1.0 * …`.
7. Năm cách giải, từ dễ hiểu đến gọn nhất:
   - **Cách 1**: Python thuần (lọc rồi nhóm bằng `dict`) để hiểu đề.
   - **Cách 2**: JOIN `Users` hai lần, rồi GROUP BY với `SUM(CASE)/COUNT` (**cách chính**).
   - **Cách 3**: Subquery `NOT IN` danh sách người bị cấm.
   - **Cách 4**: `AVG(status != 'completed')` kiểu MySQL (chạy được cả trên sqlite).
   - **Cách 5**: Pandas (tùy chọn), đúng signature LeetCode.
8. Các lỗi hay gặp (code sai, vì sao sai, cách sửa), mẹo nhận dạng pattern, biến thể, và 6 bài tập có đáp án ở cuối.

## 1. Đề bài (theo LeetCode, bản hiện hành)

Table: `Trips`

```text
+-------------+----------+
| Column Name | Type     |
+-------------+----------+
| id          | int      |
| client_id   | int      |
| driver_id   | int      |
| city_id     | int      |
| status      | enum     |
| request_at  | varchar  |
+-------------+----------+
id is the primary key (column with unique values) for this table.
The table holds all taxi trips. Each trip has a unique id, while client_id and driver_id
are foreign keys to the users_id at the Users table.
Status is an ENUM (category) type of ('completed', 'cancelled_by_driver', 'cancelled_by_client').
```

Table: `Users`

```text
+-------------+----------+
| Column Name | Type     |
+-------------+----------+
| users_id    | int      |
| banned      | enum     |
| role        | enum     |
+-------------+----------+
users_id is the primary key (column with unique values) for this table.
The table holds all users. Each user has a unique users_id, and role is an ENUM type of
('client', 'driver', 'partner').
banned is an ENUM (category) type of ('Yes', 'No').
```

The **cancellation rate** is computed by dividing the number of canceled (by client or driver) requests with unbanned users by the total number of requests with unbanned users on that day.

Write a solution to find the **cancellation rate** of requests with unbanned users (**both client and driver must not be banned**) each day between `"2013-10-01"` and `"2013-10-03"` with **at least** one trip. Round `Cancellation Rate` to **two decimal** points.

Return the result table in **any order**.

---

**Dịch nhanh.** Bảng `Trips` lưu mọi chuyến taxi: mã chuyến `id`, mã khách `client_id`, mã tài xế `driver_id`, mã thành phố `city_id`, trạng thái `status` (hoàn thành / tài xế hủy / khách hủy) và ngày đặt `request_at`. Bảng `Users` lưu mọi người dùng: mã `users_id`, có bị cấm không (`banned` = `'Yes'`/`'No'`) và vai trò `role`.

**Tỉ lệ hủy** của một ngày = (số chuyến **bị hủy**, dù khách hay tài xế hủy) chia (tổng số chuyến), **chỉ tính những chuyến mà cả khách lẫn tài xế đều không bị cấm**. Tính tỉ lệ này cho **mỗi ngày** từ `2013-10-01` đến `2013-10-03`, chỉ những ngày có ít nhất một chuyến (hợp lệ), làm tròn 2 chữ số thập phân. Thứ tự hàng kết quả không quan trọng.

> Đề này **không** phải bài Premium. Tên cột hiện hành viết **chữ thường kiểu snake_case** (`client_id`, `request_at`, `users_id`…). Nhiều lời giải cũ trên mạng viết `Client_Id`, `Banned`, `Request_at` vì đề cũ dùng tên viết hoa. MySQL không phân biệt hoa thường ở **tên cột** nên các lời giải đó vẫn chạy, nhưng bạn nên viết đúng tên hiện hành.

### Example 1 (đúng như trên LeetCode)

```text
Input:
Trips table:
+----+-----------+-----------+---------+---------------------+------------+
| id | client_id | driver_id | city_id | status              | request_at |
+----+-----------+-----------+---------+---------------------+------------+
| 1  | 1         | 10        | 1       | completed           | 2013-10-01 |
| 2  | 2         | 11        | 1       | cancelled_by_driver | 2013-10-01 |
| 3  | 3         | 12        | 6       | completed           | 2013-10-01 |
| 4  | 4         | 13        | 6       | cancelled_by_client | 2013-10-01 |
| 5  | 1         | 10        | 1       | completed           | 2013-10-02 |
| 6  | 2         | 11        | 6       | completed           | 2013-10-02 |
| 7  | 3         | 12        | 6       | completed           | 2013-10-02 |
| 8  | 2         | 12        | 12      | completed           | 2013-10-03 |
| 9  | 3         | 10        | 12      | completed           | 2013-10-03 |
| 10 | 4         | 13        | 12      | cancelled_by_driver | 2013-10-03 |
+----+-----------+-----------+---------+---------------------+------------+
Users table:
+----------+--------+--------+
| users_id | banned | role   |
+----------+--------+--------+
| 1        | No     | client |
| 2        | Yes    | client |
| 3        | No     | client |
| 4        | No     | client |
| 10       | No     | driver |
| 11       | No     | driver |
| 12       | No     | driver |
| 13       | No     | driver |
+----------+--------+--------+
Output:
+------------+-------------------+
| Day        | Cancellation Rate |
+------------+-------------------+
| 2013-10-01 | 0.33              |
| 2013-10-02 | 0.00              |
| 2013-10-03 | 0.50              |
+------------+-------------------+
```

**Giải thích của đề (dịch):**

- **2013-10-01**: có 4 chuyến, 2 chuyến bị hủy. Nhưng chuyến `id=2` do khách **bị cấm** (`users_id=2`) đặt, nên bị bỏ qua. Còn lại 3 chuyến hợp lệ, 1 chuyến bị hủy (chuyến 4), nên tỉ lệ là `1/3 = 0.33`.
- **2013-10-02**: có 3 chuyến, không chuyến nào bị hủy. Chuyến `id=6` của khách bị cấm bị bỏ. Còn 2 chuyến hợp lệ, 0 chuyến hủy, nên tỉ lệ là `0/2 = 0.00`.
- **2013-10-03**: có 3 chuyến, 1 chuyến bị hủy. Chuyến `id=8` của khách bị cấm bị bỏ. Còn 2 chuyến hợp lệ, 1 chuyến hủy (chuyến 10), nên tỉ lệ là `1/2 = 0.50`.

Để ý: trong ví dụ này **không có tài xế nào bị cấm**. Nghĩa là một lời giải **quên lọc tài xế** vẫn ra đúng Example 1 nhưng sẽ sai ở test ẩn. Đây là cái bẫy số 1 của bài.

## 2. Đọc đề thật chậm: mỗi cụm từ là một điều kiện

Với bài SQL, cách đọc đề hiệu quả nhất là **gạch chân từng cụm từ** rồi hỏi: "cụm này sẽ biến thành phần nào của câu truy vấn?"

| Cụm từ trong đề | Nghĩa | Sẽ thành gì trong lời giải |
|---|---|---|
| *requests with unbanned users* | chỉ tính chuyến của người **không bị cấm** | lọc `banned = 'No'` |
| *both client and driver must not be banned* | phải kiểm tra **hai người**: khách **và** tài xế | lọc 2 lần: một cho `client_id`, một cho `driver_id` |
| *canceled (by client or driver)* | `cancelled_by_client` **hoặc** `cancelled_by_driver` | `status != 'completed'` |
| *each day* | mỗi ngày một hàng kết quả | gom nhóm theo `request_at` |
| *between "2013-10-01" and "2013-10-03"* | lấy cả hai đầu mút | `request_at BETWEEN '2013-10-01' AND '2013-10-03'` |
| *with at least one trip* | ngày không có chuyến (hợp lệ) nào thì **không** in ra | tự nhiên đúng khi lọc trước rồi mới gom nhóm |
| *Round … to two decimal points* | làm tròn 2 chữ số thập phân | `ROUND(…, 2)` |
| *Return … in any order* | thứ tự hàng không quan trọng | không cần `ORDER BY` |
| tên cột `Day`, `Cancellation Rate` | phải đặt **đúng tên**, có cả dấu cách | `AS Day`, ``AS `Cancellation Rate` `` |

**Một chỗ cần hiểu cho đúng:** "with at least one trip" được hiểu (và được chấm) là ngày có **ít nhất một chuyến hợp lệ**. Nếu một ngày có chuyến nhưng **tất cả** đều dính người bị cấm thì mẫu số bằng 0, không có tỉ lệ nào để in. Lời giải chuẩn chỉ đơn giản là ngày đó **không xuất hiện**, và điều này xảy ra tự động khi ta **lọc trước, gom nhóm sau**.

Toàn bộ bài toán gói lại trong 4 bước:

```text
(1) bỏ chuyến của người bị cấm  ->  (2) bỏ chuyến ngoài khoảng ngày
->  (3) gom các chuyến còn lại theo ngày  ->  (4) mỗi ngày: số hủy / tổng, làm tròn 2 chữ số
```

Trước khi viết SQL, ta cần học những viên gạch nền.

## 3. Nền tảng 1: cơ sở dữ liệu quan hệ là gì?

Hãy tưởng tượng một **file Excel** có nhiều sheet. Mỗi sheet là một **bảng (table)**.

- **Bảng (table)**: một tập dữ liệu cùng loại, ví dụ bảng `Users` chứa người dùng.
- **Hàng (row, còn gọi là record)**: một đối tượng cụ thể, ví dụ người dùng số 2.
- **Cột (column, còn gọi là field)**: một thuộc tính, ví dụ cột `banned`. Mỗi cột có **kiểu dữ liệu**: `int` (số nguyên), `varchar` (chuỗi), `enum` (chỉ được chọn trong một danh sách giá trị cố định).

```text
Bảng Users
+----------+--------+--------+   <- tên các cột
| users_id | banned | role   |
+----------+--------+--------+
| 1        | No     | client |   <- một hàng = một người dùng
| 2        | Yes    | client |
+----------+--------+--------+
```

**Khóa chính (primary key)** là cột mà mỗi hàng có một giá trị **khác nhau**, dùng để chỉ đích danh một hàng. Ở đây `Users.users_id` và `Trips.id` là khóa chính: không có hai người dùng nào trùng `users_id`.

**Khóa ngoại (foreign key)** là cột **trỏ sang khóa chính của bảng khác**. Đề nói `client_id` và `driver_id` là khóa ngoại tới `Users.users_id`. Nghĩa là:

- `Trips.client_id = 2` có nghĩa là "khách của chuyến này là **người dùng số 2** trong bảng `Users`".
- `Trips.driver_id = 11` có nghĩa là "tài xế của chuyến này là **người dùng số 11**".

Vì sao không ghi luôn "khách bị cấm hay không" vào bảng `Trips`? Vì một người dùng tham gia **rất nhiều chuyến**. Nếu ghi lặp lại thì khi người đó bị cấm, ta phải sửa hàng nghìn hàng. Ghi một lần ở `Users` rồi **tham chiếu** bằng mã là cách tổ chức gọn và không mâu thuẫn. Cái giá phải trả: khi cần biết "khách của chuyến 2 có bị cấm không", ta phải **ghép (JOIN)** hai bảng. Đó chính là trọng tâm của bài này.

### `sqlite3`: một cơ sở dữ liệu mini có sẵn trong Python

Trên LeetCode, query của bạn chạy trên **MySQL**. Trong notebook, ta dùng **SQLite** qua module `sqlite3` của thư viện chuẩn Python: không cần cài gì, tạo được cơ sở dữ liệu ngay **trong RAM** (`":memory:"`).

Ba thao tác cần biết:

| Lệnh Python | Ý nghĩa |
|---|---|
| `conn = sqlite3.connect(":memory:")` | mở một cơ sở dữ liệu trống trong RAM (`conn` = connection, kết nối) |
| `cur = conn.execute("câu SQL")` | gửi một câu SQL; nhận về `cur` (cursor, con trỏ kết quả) |
| `cur.fetchall()` | lấy hết các hàng kết quả, mỗi hàng là một `tuple` |

Ngoài ra `cur.description` cho biết **tên các cột** của kết quả (phần tử đầu của mỗi mục).

Phần lõi của SQL (SELECT, WHERE, JOIN, GROUP BY…) giống nhau giữa SQLite và MySQL. Notebook sẽ chỉ rõ từng chỗ **khác nhau** khi gặp: kiểu `ENUM`, **phép chia số nguyên**, cách **làm tròn** ở ca "đúng nửa", cách **quote tên cột**.

Thử ngay một bảng mini về thú cưng:

In [1]:
import sqlite3

print("Phiên bản SQLite:", sqlite3.sqlite_version)

demo = sqlite3.connect(":memory:")                 # cơ sở dữ liệu nằm trong RAM
demo.execute("CREATE TABLE pets (name TEXT, age INTEGER)")
demo.execute("INSERT INTO pets VALUES ('Mít', 3)")
demo.execute("INSERT INTO pets VALUES ('Na', 5)")

cur = demo.execute("SELECT name, age FROM pets")
print("Các hàng:", cur.fetchall())                 # list các tuple
print("Tên cột :", [col[0] for col in cur.description])

Phiên bản SQLite: 3.53.1
Các hàng: [('Mít', 3), ('Na', 5)]
Tên cột : ['name', 'age']


### Hàm tiện ích: in kết quả thành bảng giống LeetCode

Một list tuple như `[('Mít', 3), ('Na', 5)]` khó đọc khi bảng dài. Ta viết 3 hàm nhỏ, dùng suốt notebook:

- `run_query(conn, sql)`: chạy query, trả về `(danh sách tên cột, danh sách hàng)`.
- `print_table(columns, rows)`: in đẹp có khung `+----+`, giá trị `None` của Python (tức `NULL` trong SQL) in thành chữ `NULL`.
- `show(conn, sql)`: chạy rồi in luôn.

Thêm hàm `normalize(rows)`: **sắp xếp** các hàng và **làm tròn** số thực tới 2 chữ số. Vì đề nói "any order", khi so kết quả với đáp án ta luôn sắp xếp trước rồi mới so.

In [2]:
def run_query(conn, sql, params=()):
    cur = conn.execute(sql, params)
    columns = [col[0] for col in cur.description] if cur.description else []
    return columns, cur.fetchall()


def print_table(columns, rows):
    text_rows = [["NULL" if v is None else str(v) for v in row] for row in rows]
    widths = [len(name) for name in columns]
    for row in text_rows:
        for j, value in enumerate(row):
            widths[j] = max(widths[j], len(value))
    line = "+" + "+".join("-" * (w + 2) for w in widths) + "+"
    print(line)
    print("| " + " | ".join(name.ljust(w) for name, w in zip(columns, widths)) + " |")
    print(line)
    for row in text_rows:
        print("| " + " | ".join(v.ljust(w) for v, w in zip(row, widths)) + " |")
    print(line, f"({len(rows)} hàng)")


def show(conn, sql):
    columns, rows = run_query(conn, sql)
    print_table(columns, rows)


def normalize(rows):
    """Làm tròn số thực tới 2 chữ số rồi sắp xếp, để so sánh không phụ thuộc thứ tự hàng."""
    result = []
    for row in rows:
        result.append(tuple(round(float(v), 2) if isinstance(v, float) else v for v in row))
    return sorted(result)


show(demo, "SELECT name, age FROM pets")
print(normalize([("b", 0.333333), ("a", 0.5)]))

+------+-----+
| name | age |
+------+-----+
| Mít  | 3   |
| Na   | 5   |
+------+-----+ (2 hàng)
[('a', 0.5), ('b', 0.33)]


### Tạo đúng hai bảng của đề trong SQLite

Trên LeetCode, schema MySQL của bài là:

```sql
Create table If Not Exists Trips (id int, client_id int, driver_id int, city_id int,
    status ENUM('completed', 'cancelled_by_driver', 'cancelled_by_client'), request_at varchar(50))
Create table If Not Exists Users (users_id int, banned varchar(50), role ENUM('client', 'driver', 'partner'))
```

SQLite **không có kiểu `ENUM`**. Ta thay bằng `TEXT` cộng một ràng buộc `CHECK (status IN (...))`: nếu ai đó chèn giá trị lạ, SQLite báo lỗi ngay. `varchar` đổi thành `TEXT`, `int` đổi thành `INTEGER`. Ta cũng khai báo `PRIMARY KEY` đúng như đề mô tả.

Đọc câu `CREATE TABLE` như sau:

```sql
CREATE TABLE Trips (            -- tạo bảng tên Trips gồm các cột:
    id INTEGER PRIMARY KEY,     --   id: số nguyên, là khóa chính
    status TEXT CHECK (...),    --   status: chuỗi, chỉ nhận 3 giá trị cho phép
    ...
);
```

Để chèn dữ liệu, ta dùng `INSERT INTO Trips VALUES (?, ?, ?, ?, ?, ?)`. Mỗi dấu `?` là **chỗ trống** để Python điền giá trị vào (parameter binding, gắn tham số): an toàn và khỏi lo chuyện dấu nháy. `executemany` chèn cả một list tuple trong một lệnh, còn `executescript` chạy nhiều câu SQL cách nhau bởi `;`.

Hàm `build_db(trips_rows, users_rows)` dưới đây tạo một cơ sở dữ liệu **mới tinh** với hai bảng và dữ liệu đưa vào. Ta sẽ gọi nó rất nhiều lần, mỗi bộ test một cơ sở dữ liệu riêng.

In [3]:
SCHEMA = """
CREATE TABLE Trips (
    id          INTEGER PRIMARY KEY,
    client_id   INTEGER,
    driver_id   INTEGER,
    city_id     INTEGER,
    status      TEXT CHECK (status IN ('completed', 'cancelled_by_driver', 'cancelled_by_client')),
    request_at  TEXT
);
CREATE TABLE Users (
    users_id    INTEGER PRIMARY KEY,
    banned      TEXT CHECK (banned IN ('Yes', 'No')),
    role        TEXT CHECK (role IN ('client', 'driver', 'partner'))
);
"""


def build_db(trips_rows, users_rows):
    conn = sqlite3.connect(":memory:")
    conn.executescript(SCHEMA)
    conn.executemany("INSERT INTO Trips VALUES (?, ?, ?, ?, ?, ?)", trips_rows)
    conn.executemany("INSERT INTO Users VALUES (?, ?, ?)", users_rows)
    return conn


# Dữ liệu Example 1, chép đúng từ đề
TRIPS_EX1 = [
    (1, 1, 10, 1, "completed", "2013-10-01"),
    (2, 2, 11, 1, "cancelled_by_driver", "2013-10-01"),
    (3, 3, 12, 6, "completed", "2013-10-01"),
    (4, 4, 13, 6, "cancelled_by_client", "2013-10-01"),
    (5, 1, 10, 1, "completed", "2013-10-02"),
    (6, 2, 11, 6, "completed", "2013-10-02"),
    (7, 3, 12, 6, "completed", "2013-10-02"),
    (8, 2, 12, 12, "completed", "2013-10-03"),
    (9, 3, 10, 12, "completed", "2013-10-03"),
    (10, 4, 13, 12, "cancelled_by_driver", "2013-10-03"),
]
USERS_EX1 = [
    (1, "No", "client"), (2, "Yes", "client"), (3, "No", "client"), (4, "No", "client"),
    (10, "No", "driver"), (11, "No", "driver"), (12, "No", "driver"), (13, "No", "driver"),
]
EXPECTED_EX1 = [("2013-10-01", 0.33), ("2013-10-02", 0.0), ("2013-10-03", 0.5)]

conn_ex1 = build_db(TRIPS_EX1, USERS_EX1)

# CHECK hoạt động như ENUM: chèn trạng thái lạ sẽ bị từ chối
try:
    conn_ex1.execute("INSERT INTO Trips VALUES (99, 1, 10, 1, 'lost', '2013-10-01')")
except sqlite3.IntegrityError as error:
    print("Bị từ chối đúng như mong đợi:", error)

show(conn_ex1, "SELECT * FROM Users")

Bị từ chối đúng như mong đợi: CHECK constraint failed: status IN ('completed', 'cancelled_by_driver', 'cancelled_by_client')
+----------+--------+--------+
| users_id | banned | role   |
+----------+--------+--------+
| 1        | No     | client |
| 2        | Yes    | client |
| 3        | No     | client |
| 4        | No     | client |
| 10       | No     | driver |
| 11       | No     | driver |
| 12       | No     | driver |
| 13       | No     | driver |
+----------+--------+--------+ (8 hàng)


## 4. Nền tảng 2: SELECT, WHERE, ORDER BY, LIMIT

Câu truy vấn SQL đọc **gần như một câu tiếng Anh**:

```sql
SELECT id, status          -- lấy những cột nào
FROM Trips                 -- từ bảng nào
WHERE status != 'completed'-- chỉ giữ những hàng thỏa điều kiện
ORDER BY id DESC           -- sắp xếp theo cột nào (DESC = giảm dần, mặc định ASC = tăng dần)
LIMIT 3;                   -- chỉ lấy tối đa 3 hàng
```

- `SELECT *` nghĩa là lấy **tất cả** các cột.
- `AS` đặt **tên mới** (alias) cho cột kết quả: `SELECT client_id AS khach`.
- Chuỗi trong SQL viết trong **dấu nháy đơn**: `'completed'`, `'No'`.
- `WHERE` dùng các phép so sánh `=` (bằng; chỉ **một** dấu `=`, khác Python), `!=` hoặc `<>` (khác), `<`, `>`, `<=`, `>=`, nối bằng `AND`, `OR`, `NOT`.

Lưu ý nhỏ: SQLite so sánh chuỗi **phân biệt hoa thường** (`'no'` khác `'No'`), còn MySQL với cấu hình mặc định thì **không** phân biệt. Cứ viết đúng y như dữ liệu (`'Yes'`, `'No'`) là an toàn ở cả hai nơi.

Ba query nhỏ trên Example 1:

In [4]:
print("3 chuyến có id lớn nhất:")
show(conn_ex1, "SELECT id, client_id AS khach, status FROM Trips ORDER BY id DESC LIMIT 3")

print("\nNgười dùng bị cấm:")
show(conn_ex1, "SELECT users_id, role FROM Users WHERE banned = 'Yes'")

print("\nChuyến bị hủy trong ngày 2013-10-01:")
show(conn_ex1, """
SELECT id, status
FROM Trips
WHERE status != 'completed' AND request_at = '2013-10-01'
""")

3 chuyến có id lớn nhất:
+----+-------+---------------------+
| id | khach | status              |
+----+-------+---------------------+
| 10 | 4     | cancelled_by_driver |
| 9  | 3     | completed           |
| 8  | 2     | completed           |
+----+-------+---------------------+ (3 hàng)

Người dùng bị cấm:
+----------+--------+
| users_id | role   |
+----------+--------+
| 2        | client |
+----------+--------+ (1 hàng)

Chuyến bị hủy trong ngày 2013-10-01:
+----+---------------------+
| id | status              |
+----+---------------------+
| 2  | cancelled_by_driver |
| 4  | cancelled_by_client |
+----+---------------------+ (2 hàng)


### BETWEEN và "ngày" lưu dạng chuỗi

Cột `request_at` có kiểu `varchar`, tức là **chuỗi**, không phải kiểu ngày tháng. Vậy so sánh `request_at` với `'2013-10-03'` có đúng thứ tự thời gian không?

**Có**, nhờ định dạng `YYYY-MM-DD` luôn đủ chữ số (tháng và ngày có số 0 đứng đầu). Hai chuỗi được so **từng ký tự từ trái sang phải** (thứ tự từ điển, lexicographic order). Ví dụ `'2013-10-02'` và `'2013-10-10'` giống nhau ở 8 ký tự đầu `2013-10-`, tới ký tự thứ 9 thì `'0' < '1'`, nên `'2013-10-02' < '2013-10-10'`, khớp với thời gian.

Nếu ai đó lưu `'2013-10-2'` (thiếu số 0) thì thứ tự **hỏng**: ký tự thứ 9 là `'2' > '1'`, nên `'2013-10-2'` lại "lớn hơn" `'2013-10-10'`. Đề này dữ liệu luôn đúng định dạng nên ta yên tâm.

`x BETWEEN a AND b` nghĩa là `a <= x AND x <= b`, **lấy cả hai đầu mút**. Vì vậy `BETWEEN '2013-10-01' AND '2013-10-03'` lấy đủ cả ngày 1 lẫn ngày 3.

In [5]:
print("'2013-10-02' < '2013-10-10' ?", "2013-10-02" < "2013-10-10")   # True, đúng thứ tự thời gian
print("'2013-10-2'  < '2013-10-10' ?", "2013-10-2" < "2013-10-10")    # False! sai định dạng làm hỏng thứ tự

print("\nBETWEEN lấy cả hai đầu mút (ngày 02 và ngày 03 đều có mặt):")
show(conn_ex1, """
SELECT id, request_at
FROM Trips
WHERE request_at BETWEEN '2013-10-02' AND '2013-10-03'
ORDER BY id
""")

'2013-10-02' < '2013-10-10' ? True
'2013-10-2'  < '2013-10-10' ? False

BETWEEN lấy cả hai đầu mút (ngày 02 và ngày 03 đều có mặt):
+----+------------+
| id | request_at |
+----+------------+
| 5  | 2013-10-02 |
| 6  | 2013-10-02 |
| 7  | 2013-10-02 |
| 8  | 2013-10-03 |
| 9  | 2013-10-03 |
| 10 | 2013-10-03 |
+----+------------+ (6 hàng)


## 5. Nền tảng 3: JOIN, ghép hai bảng lại

Bảng `Trips` chỉ biết **mã** khách (`client_id = 2`), không biết khách đó có bị cấm không. Thông tin `banned` nằm ở bảng `Users`. Ta cần **ghép** mỗi chuyến với đúng hàng người dùng tương ứng.

```sql
SELECT t.id, t.client_id, c.banned
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id
```

Đọc từng phần:

- `Trips AS t`: đặt **tên tắt (alias)** `t` cho bảng `Trips`. Từ đó `t.client_id` nghĩa là "cột `client_id` của bảng `t`". Chữ `AS` có thể bỏ: viết `Trips t` cũng được.
- `JOIN Users AS c`: ghép thêm bảng `Users`, đặt tên tắt `c` (viết tắt của *client*).
- `ON t.client_id = c.users_id`: **điều kiện ghép**. Với mỗi hàng của `t`, tìm hàng của `c` có `users_id` bằng `client_id` của chuyến đó.

`JOIN` không ghi gì thêm chính là **INNER JOIN**: chỉ giữ những cặp hàng **khớp** được điều kiện `ON`. Vì `users_id` là khóa chính (không trùng), mỗi chuyến khớp **đúng một** người dùng, nên số hàng sau khi JOIN vẫn là 10, chỉ là mỗi hàng có thêm cột `banned` của khách.

Cách hình dung: với chuyến 2, máy "tra danh bạ" `Users` tìm người số 2, thấy `banned = 'Yes'`, rồi dán thông tin đó vào cạnh hàng chuyến 2.

In [6]:
show(conn_ex1, """
SELECT t.id, t.client_id, c.banned AS client_banned
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id
ORDER BY t.id
""")

+----+-----------+---------------+
| id | client_id | client_banned |
+----+-----------+---------------+
| 1  | 1         | No            |
| 2  | 2         | Yes           |
| 3  | 3         | No            |
| 4  | 4         | No            |
| 5  | 1         | No            |
| 6  | 2         | Yes           |
| 7  | 3         | No            |
| 8  | 2         | Yes           |
| 9  | 3         | No            |
| 10 | 4         | No            |
+----+-----------+---------------+ (10 hàng)


### JOIN cùng một bảng **hai lần**: một cho khách, một cho tài xế

Mỗi chuyến có **hai** người: khách và tài xế, và **cả hai** đều nằm trong cùng bảng `Users`. Bảng `Users` vì vậy đóng **hai vai trò**. Ta JOIN nó hai lần với **hai alias khác nhau**:

```sql
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id   -- bản "danh bạ" thứ nhất: tra khách
JOIN Users AS d ON t.driver_id = d.users_id   -- bản "danh bạ" thứ hai: tra tài xế
```

Hãy tưởng tượng bạn **photo** sổ danh bạ thành hai bản: một bản dán nhãn `c` để tra khách, một bản dán nhãn `d` để tra tài xế. Sau hai lần JOIN, mỗi hàng chuyến có cả `c.banned` (khách bị cấm?) lẫn `d.banned` (tài xế bị cấm?).

Alias ở đây là **bắt buộc**: nếu viết `Users` hai lần mà không đặt tên khác nhau, SQL không biết `Users.banned` là của khách hay của tài xế.

Vì sao không JOIN **một lần** với điều kiện `ON t.client_id = u.users_id OR t.driver_id = u.users_id`? Khi đó mỗi chuyến khớp **hai** hàng người dùng (khách và tài xế), nên số hàng **nhân đôi** và việc lọc bị sai. Phần "Lỗi thường gặp" sẽ chạy thử để bạn thấy tận mắt.

In [7]:
show(conn_ex1, """
SELECT t.id,
       t.client_id, c.banned AS client_banned,
       t.driver_id, d.banned AS driver_banned
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id
JOIN Users AS d ON t.driver_id = d.users_id
ORDER BY t.id
""")

+----+-----------+---------------+-----------+---------------+
| id | client_id | client_banned | driver_id | driver_banned |
+----+-----------+---------------+-----------+---------------+
| 1  | 1         | No            | 10        | No            |
| 2  | 2         | Yes           | 11        | No            |
| 3  | 3         | No            | 12        | No            |
| 4  | 4         | No            | 13        | No            |
| 5  | 1         | No            | 10        | No            |
| 6  | 2         | Yes           | 11        | No            |
| 7  | 3         | No            | 12        | No            |
| 8  | 2         | Yes           | 12        | No            |
| 9  | 3         | No            | 10        | No            |
| 10 | 4         | No            | 13        | No            |
+----+-----------+---------------+-----------+---------------+ (10 hàng)


### INNER JOIN và LEFT JOIN khác nhau thế nào?

Giả sử có một chuyến mà khách **không có** trong bảng `Users` (dữ liệu lỗi, ví dụ `client_id = 99`):

- **INNER JOIN** (`JOIN`): chuyến đó **không khớp** với ai nên **biến mất** khỏi kết quả.
- **LEFT JOIN**: **giữ nguyên mọi hàng của bảng bên trái** (`Trips`). Hàng nào không khớp thì các cột của bảng bên phải được điền `NULL` (giá trị "không có / không biết").

Với đề này, khóa ngoại đảm bảo mọi `client_id`, `driver_id` đều có trong `Users`, nên INNER JOIN là đủ và tự nhiên nhất. Biết LEFT JOIN vẫn quan trọng vì nhiều người dùng LEFT JOIN rồi đặt điều kiện `banned = 'No'` vào `ON`, và đó là một lỗi rất khó thấy (xem mục Lỗi 4). Thử trên một cơ sở dữ liệu mini:

In [8]:
mini = build_db(
    trips_rows=[(1, 1, 10, 1, "completed", "2013-10-01"),
                (2, 99, 10, 1, "completed", "2013-10-01")],   # khách 99 không có trong Users
    users_rows=[(1, "No", "client"), (10, "No", "driver")],
)
print("INNER JOIN: chuyến 2 biến mất")
show(mini, "SELECT t.id, t.client_id, c.banned FROM Trips t JOIN Users c ON t.client_id = c.users_id")
print("\nLEFT JOIN: chuyến 2 vẫn còn, cột banned là NULL")
show(mini, "SELECT t.id, t.client_id, c.banned FROM Trips t LEFT JOIN Users c ON t.client_id = c.users_id")

INNER JOIN: chuyến 2 biến mất
+----+-----------+--------+
| id | client_id | banned |
+----+-----------+--------+
| 1  | 1         | No     |
+----+-----------+--------+ (1 hàng)

LEFT JOIN: chuyến 2 vẫn còn, cột banned là NULL
+----+-----------+--------+
| id | client_id | banned |
+----+-----------+--------+
| 1  | 1         | No     |
| 2  | 99        | NULL   |
+----+-----------+--------+ (2 hàng)


## 6. Nền tảng 4: subquery, IN và NOT IN

**Subquery** (truy vấn con) là một câu `SELECT` nằm **bên trong** câu `SELECT` khác, đặt trong ngoặc đơn. Ví dụ:

```sql
SELECT users_id FROM Users WHERE banned = 'Yes'     -- trả về danh sách: 2
```

Kết quả của nó là một **danh sách mã người bị cấm**. Ta dùng danh sách này để lọc:

```sql
WHERE client_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
```

đọc là "giữ chuyến nếu mã khách **không nằm trong** danh sách người bị cấm". `x IN (a, b, c)` giống `x == a or x == b or x == c` trong Python; `NOT IN` là phủ định của nó.

### Cái bẫy `NULL` của `NOT IN`

`NULL` trong SQL nghĩa là "không biết". So sánh bất cứ thứ gì với `NULL` đều ra **không biết** (cũng là `NULL`), không ra đúng hay sai. Vì vậy:

```text
5 NOT IN (1, NULL)   =   (5 <> 1) AND (5 <> NULL)   =   TRUE AND NULL   =   NULL
```

`WHERE` chỉ giữ hàng khi điều kiện **đúng hẳn**, nên hàng đó bị **loại**. Chỉ cần danh sách con có **một** giá trị `NULL` là `NOT IN` loại sạch mọi hàng. Ở bài này `users_id` là khóa chính nên không bao giờ `NULL`, ta an toàn. Nhưng hãy tập thói quen: khi dùng `NOT IN (subquery)`, tự hỏi "subquery có thể trả về `NULL` không?". Nếu có thì thêm `WHERE ... IS NOT NULL` vào subquery, hoặc dùng `NOT EXISTS`.

In [9]:
print("Subquery chạy riêng: danh sách người bị cấm")
show(conn_ex1, "SELECT users_id FROM Users WHERE banned = 'Yes'")

print("\nChỉ giữ chuyến mà cả khách lẫn tài xế KHÔNG nằm trong danh sách đó:")
show(conn_ex1, """
SELECT id, client_id, driver_id, request_at
FROM Trips
WHERE client_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND driver_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
""")

print("\nBẫy NULL: cột c ra NULL chứ không ra 1")
show(demo, "SELECT 5 IN (1, 5) AS a, 5 NOT IN (1, 2) AS b, 5 NOT IN (1, NULL) AS c")

Subquery chạy riêng: danh sách người bị cấm
+----------+
| users_id |
+----------+
| 2        |
+----------+ (1 hàng)

Chỉ giữ chuyến mà cả khách lẫn tài xế KHÔNG nằm trong danh sách đó:
+----+-----------+-----------+------------+
| id | client_id | driver_id | request_at |
+----+-----------+-----------+------------+
| 1  | 1         | 10        | 2013-10-01 |
| 3  | 3         | 12        | 2013-10-01 |
| 4  | 4         | 13        | 2013-10-01 |
| 5  | 1         | 10        | 2013-10-02 |
| 7  | 3         | 12        | 2013-10-02 |
| 9  | 3         | 10        | 2013-10-03 |
| 10 | 4         | 13        | 2013-10-03 |
+----+-----------+-----------+------------+ (7 hàng)

Bẫy NULL: cột c ra NULL chứ không ra 1
+---+---+------+
| a | b | c    |
+---+---+------+
| 1 | 1 | NULL |
+---+---+------+ (1 hàng)


## 7. Nền tảng 5: GROUP BY và COUNT, gom nhóm rồi đếm

Đề hỏi "**mỗi ngày**" nên ta phải **gom** các chuyến cùng ngày lại thành một nhóm, rồi tính toán trên từng nhóm. Đó là việc của `GROUP BY`:

```sql
SELECT request_at, COUNT(*) AS so_chuyen
FROM Trips
GROUP BY request_at
```

- `GROUP BY request_at`: các hàng có cùng `request_at` được gom vào một nhóm. Kết quả có **một hàng cho mỗi nhóm**.
- `COUNT(*)`: đếm số hàng trong nhóm. Đây là một **hàm gộp (aggregate function)**: nhận cả nhóm, trả về một con số. Các hàm gộp khác: `SUM` (tổng), `AVG` (trung bình), `MIN`, `MAX`.
- `GROUP_CONCAT(id)`: nối các giá trị trong nhóm thành một chuỗi `"1,2,3"`. Có cả trong MySQL lẫn SQLite, rất tiện để **nhìn xem** mỗi nhóm chứa hàng nào.

**Quy tắc:** sau khi `GROUP BY`, trong `SELECT` chỉ được ghi (a) cột dùng để nhóm, hoặc (b) hàm gộp. Ghi một cột khác như `status` là vô nghĩa, vì một nhóm có nhiều `status` khác nhau thì biết lấy cái nào?

**Thứ tự máy "thực hiện" một câu truy vấn** (thứ tự logic, khác thứ tự ta viết):

```text
FROM + JOIN  ->  WHERE  ->  GROUP BY  ->  (tính hàm gộp)  ->  SELECT  ->  ORDER BY  ->  LIMIT
```

Vì `WHERE` chạy **trước** `GROUP BY`, những chuyến bị loại ở `WHERE` sẽ không bao giờ vào nhóm nào. Đây là lý do "ngày không có chuyến hợp lệ" tự động biến mất khỏi kết quả.

In [10]:
show(conn_ex1, """
SELECT request_at, COUNT(*) AS so_chuyen, GROUP_CONCAT(id) AS cac_id
FROM Trips
GROUP BY request_at
""")

+------------+-----------+---------+
| request_at | so_chuyen | cac_id  |
+------------+-----------+---------+
| 2013-10-01 | 4         | 1,2,3,4 |
| 2013-10-02 | 3         | 5,6,7   |
| 2013-10-03 | 3         | 8,9,10  |
+------------+-----------+---------+ (3 hàng)


### Đếm có điều kiện: `SUM(CASE WHEN ...)` và biểu thức đúng/sai là số 0/1

`COUNT(*)` đếm **mọi** hàng. Nhưng ta còn cần đếm **riêng các chuyến bị hủy** trong mỗi nhóm. Mẹo: biến mỗi hàng thành **1** nếu bị hủy, **0** nếu không, rồi **cộng lại**.

`CASE WHEN` là câu `if/else` của SQL:

```sql
CASE WHEN status != 'completed' THEN 1 ELSE 0 END
```

tương đương Python `1 if status != "completed" else 0`. Sau đó `SUM(CASE WHEN ... THEN 1 ELSE 0 END)` là tổng các số 1, tức **số hàng thỏa điều kiện**.

**Biết thêm:** trong cả **MySQL** lẫn **SQLite**, bản thân phép so sánh `status != 'completed'` đã trả về **1** (đúng) hoặc **0** (sai). Nên viết `SUM(status != 'completed')` cũng ra kết quả như `SUM(CASE ...)`. Cách `CASE` dài hơn nhưng chạy được ở **mọi** hệ quản trị (PostgreSQL có kiểu `boolean` riêng, không cộng trực tiếp được). Tính chất "đúng = 1, sai = 0" sẽ là chìa khóa của Cách 4.

In [11]:
print("Từng hàng ngày 2013-10-01: CASE và phép so sánh cho cùng giá trị 0/1")
show(conn_ex1, """
SELECT id, status,
       CASE WHEN status != 'completed' THEN 1 ELSE 0 END AS bang_case,
       status != 'completed'                             AS bang_so_sanh
FROM Trips
WHERE request_at = '2013-10-01'
""")

print("\nMỗi ngày: tổng số chuyến và số chuyến bị hủy (chưa lọc người bị cấm)")
show(conn_ex1, """
SELECT request_at,
       COUNT(*)                                               AS tong,
       SUM(CASE WHEN status != 'completed' THEN 1 ELSE 0 END) AS so_huy
FROM Trips
GROUP BY request_at
""")

Từng hàng ngày 2013-10-01: CASE và phép so sánh cho cùng giá trị 0/1
+----+---------------------+-----------+--------------+
| id | status              | bang_case | bang_so_sanh |
+----+---------------------+-----------+--------------+
| 1  | completed           | 0         | 0            |
| 2  | cancelled_by_driver | 1         | 1            |
| 3  | completed           | 0         | 0            |
| 4  | cancelled_by_client | 1         | 1            |
+----+---------------------+-----------+--------------+ (4 hàng)

Mỗi ngày: tổng số chuyến và số chuyến bị hủy (chưa lọc người bị cấm)
+------------+------+--------+
| request_at | tong | so_huy |
+------------+------+--------+
| 2013-10-01 | 4    | 2      |
| 2013-10-02 | 3    | 0      |
| 2013-10-03 | 3    | 1      |
+------------+------+--------+ (3 hàng)


## 8. Nền tảng 6: chia số nguyên hay số thực? (SQLite **khác** MySQL)

Đây là chỗ khiến nhiều người "chạy trên máy ra 0, nộp LeetCode lại đúng" hoặc ngược lại.

| Biểu thức | Python | SQLite | MySQL |
|---|---|---|---|
| `1 / 3` | `0.333…` | **`0`** (chia nguyên!) | `0.3333` |
| `7 / 2` | `3.5` | **`3`** | `3.5000` |
| `1.0 * 1 / 3` | `0.333…` | `0.333…` | `0.3333` |
| `1 / 3 * 1.0` | `0.333…` | **`0.0`** (chia nguyên **trước**, nhân 1.0 sau) | `0.3333` |

- **MySQL**: toán tử `/` **luôn** cho số thập phân. Muốn chia nguyên thì phải dùng `DIV` (`7 DIV 2 = 3`).
- **SQLite** (và cả PostgreSQL, SQL Server): nếu **cả hai vế đều là số nguyên** thì `/` là **chia nguyên**, giống `//` của Python.

`SUM(...)` và `COUNT(*)` đều trả về số nguyên, nên `SUM(...) / COUNT(*)` trong SQLite cho ra **0** với mọi tỉ lệ nhỏ hơn 1!

**Cách sửa:** biến **một vế thành số thực trước khi chia**: `1.0 * SUM(...) / COUNT(*)`. SQL tính từ trái sang phải: `1.0 * SUM(...)` ra số thực trước, rồi mới chia. Đặt `* 1.0` ở **cuối** là vô ích, vì phép chia nguyên đã xảy ra rồi. Cách khác: `CAST(SUM(...) AS REAL)` (SQLite) hoặc `CAST(... AS DECIMAL(10, 4))` (MySQL).

Viết `1.0 * …` thì **an toàn ở mọi nơi**: trên MySQL nó không làm sai gì, trên SQLite nó cứu bạn khỏi kết quả 0.

In [12]:
print("Python :", 1 / 3, "|", 1 // 3, "|", 7 / 2, "|", 7 // 2)
show(demo, """
SELECT 1 / 3               AS a,
       7 / 2               AS b,
       1.0 * 1 / 3         AS c,
       1 / 3 * 1.0         AS d,
       CAST(1 AS REAL) / 3 AS e
""")

Python : 0.3333333333333333 | 0 | 3.5 | 3
+---+---+--------------------+-----+--------------------+
| a | b | c                  | d   | e                  |
+---+---+--------------------+-----+--------------------+
| 0 | 3 | 0.3333333333333333 | 0.0 | 0.3333333333333333 |
+---+---+--------------------+-----+--------------------+ (1 hàng)


### `AVG` và `ROUND`: trung bình và làm tròn

**`AVG(x)`** = tổng các giá trị chia số giá trị. Điều thú vị: nếu `x` chỉ gồm 0 và 1 thì

```text
AVG(x) = (số lượng số 1) / (tổng số hàng) = tỉ lệ hàng thỏa điều kiện
```

Ví dụ ba chuyến `[hủy, không, không]` thành `[1, 0, 0]`, trung bình là `1/3`, đúng bằng tỉ lệ hủy. Trong SQLite, `AVG` **luôn trả về số thực** (kiểu `real`) nên **không** dính chia nguyên. Trong MySQL, `AVG` của số nguyên trả về số thập phân. Cả hai đều ổn.

**`ROUND(x, 2)`** làm tròn `x` tới 2 chữ số sau dấu phẩy: `ROUND(0.3333, 2) = 0.33`, `ROUND(0.6667, 2) = 0.67`.

**Góc tinh tế: ca "đúng nửa"** như `1/8 = 0.125`. Ba nơi làm tròn **khác nhau**:

- **Python** `round(0.125, 2)` ra `0.12`, vì Python làm tròn "nửa về số chẵn" (banker's rounding).
- **SQLite** `ROUND(0.125, 2)` ra `0.13`. Nhưng `3/40 = 0.075` lưu trong máy thực ra là `0.07499999…` (số thực nhị phân không biểu diễn chính xác được), nên SQLite ra `0.07`.
- **MySQL** tính phép chia bằng kiểu **DECIMAL** (số thập phân chính xác) rồi làm tròn "nửa thì lên": `0.13` và `0.08`.

LeetCode chấm theo **MySQL**. Để bản Python thuần khớp MySQL tuyệt đối, ta tự viết hàm `round_rate(cancelled, total)` chỉ dùng **số nguyên**:

```text
round_half_up(k / n, 2) = floor(100·k/n + 1/2) / 100 = ((200·k + n) // (2·n)) / 100
```

Ví dụ `k=1, n=3`: `(200 + 3) // 6 = 203 // 6 = 33`, ra `0.33`. Ví dụ `k=1, n=8`: `208 // 16 = 13`, ra `0.13`.

Ví dụ của đề và mọi test trong notebook này **không** rơi vào ca "đúng nửa", nên mọi cách cho cùng kết quả. Cell dưới chỉ để bạn thấy sự khác biệt tồn tại.

In [13]:
def round_rate(cancelled, total):
    """cancelled/total làm tròn 2 chữ số kiểu 'nửa thì lên' (giống MySQL), tính bằng số nguyên chính xác."""
    return ((200 * cancelled + total) // (2 * total)) / 100


print("AVG trên cột 0/1 = tỉ lệ hủy của cả bảng (3 chuyến hủy / 10 chuyến):")
show(conn_ex1, "SELECT AVG(status != 'completed') AS ti_le, typeof(AVG(1)) AS kieu_cua_avg FROM Trips")

print("\nLàm tròn ở 3 nơi:")
print(f"{'phân số':>8} | {'Python round':>12} | {'SQLite ROUND':>12} | {'kiểu MySQL':>10}")
for k, n in [(1, 3), (1, 2), (1, 8), (3, 40)]:
    sqlite_value = demo.execute("SELECT ROUND(1.0 * ? / ?, 2)", (k, n)).fetchone()[0]
    print(f"{k:>4}/{n:<3} | {round(k / n, 2):>12} | {sqlite_value:>12} | {round_rate(k, n):>10}")

AVG trên cột 0/1 = tỉ lệ hủy của cả bảng (3 chuyến hủy / 10 chuyến):
+-------+--------------+
| ti_le | kieu_cua_avg |
+-------+--------------+
| 0.3   | real         |
+-------+--------------+ (1 hàng)

Làm tròn ở 3 nơi:
 phân số | Python round | SQLite ROUND | kiểu MySQL
   1/3   |         0.33 |         0.33 |       0.33
   1/2   |          0.5 |          0.5 |        0.5
   1/8   |         0.12 |         0.13 |       0.13
   3/40  |         0.07 |         0.07 |       0.08


## 9. Cách 1: Python thuần, hiểu đề bằng thứ đã quen

Trước khi viết SQL, hãy giải bằng Python với dữ liệu là **list các tuple** (mỗi tuple là một hàng của bảng). Mục đích: hiểu thật rõ **đề muốn tính gì**, và có một lời giải "chuẩn" để đối chiếu với các query SQL về sau.

Bốn bước, đúng như phân tích ở mục 2:

1. **Tập người bị cấm**: duyệt `users`, đưa `users_id` của ai có `banned == "Yes"` vào một `set`. Kiểm tra `x in set` nhanh, trung bình `O(1)`.
2. **Lọc từng chuyến**: bỏ chuyến ngoài khoảng ngày, bỏ chuyến có khách **hoặc** tài xế nằm trong tập bị cấm.
3. **Gom theo ngày bằng `dict`**: `stats[ngay] = [tong, so_huy]`. Gặp ngày mới thì khởi tạo `[0, 0]`, sau đó tăng `tong`, và tăng `so_huy` nếu chuyến không `completed`.
4. **Tính tỉ lệ**: với mỗi ngày, `round_rate(so_huy, tong)`.

Mẫu "gom nhóm bằng dict" (bước 3) là phiên bản Python của `GROUP BY`. Xem nó trên một ví dụ mini trước:

In [14]:
orders = [("2013-10-01", "completed"),
          ("2013-10-01", "cancelled_by_client"),
          ("2013-10-02", "completed")]

stats = {}                                  # ngày -> [tổng số chuyến, số chuyến hủy]
for day, status in orders:                  # mỗi tuple được "mở" thành 2 biến
    if day not in stats:
        stats[day] = [0, 0]                 # gặp ngày mới: khởi tạo bộ đếm
    stats[day][0] += 1
    if status != "completed":
        stats[day][1] += 1
    print(f"đọc ({day}, {status:<19}) -> stats = {stats}")

đọc (2013-10-01, completed          ) -> stats = {'2013-10-01': [1, 0]}
đọc (2013-10-01, cancelled_by_client) -> stats = {'2013-10-01': [2, 1]}
đọc (2013-10-02, completed          ) -> stats = {'2013-10-01': [2, 1], '2013-10-02': [1, 0]}


In [15]:
def trips_and_users_python(trips, users, start="2013-10-01", end="2013-10-03"):
    # Bước 1: tập id người bị cấm
    banned_ids = set()
    for users_id, banned, role in users:
        if banned == "Yes":
            banned_ids.add(users_id)

    # Bước 2 + 3: lọc từng chuyến rồi gom theo ngày
    stats = {}                                     # ngày -> [tổng, số hủy]
    for trip_id, client_id, driver_id, city_id, status, request_at in trips:
        if not (start <= request_at <= end):       # so sánh chuỗi 'YYYY-MM-DD'
            continue
        if client_id in banned_ids or driver_id in banned_ids:
            continue
        if request_at not in stats:
            stats[request_at] = [0, 0]
        stats[request_at][0] += 1
        if status != "completed":
            stats[request_at][1] += 1

    # Bước 4: mỗi ngày một hàng (Day, Cancellation Rate)
    result = []
    for day in sorted(stats):
        total, cancelled = stats[day]
        result.append((day, round_rate(cancelled, total)))
    return result


print(trips_and_users_python(TRIPS_EX1, USERS_EX1))
assert trips_and_users_python(TRIPS_EX1, USERS_EX1) == EXPECTED_EX1
print("Khớp Example 1!")

[('2013-10-01', 0.33), ('2013-10-02', 0.0), ('2013-10-03', 0.5)]
Khớp Example 1!


### Bộ dữ liệu "khó" để trace

Example 1 quá "hiền": không có tài xế bị cấm, không có chuyến ngoài khoảng ngày. Ta tự dựng một bộ dữ liệu có **đủ mọi nhánh đặc biệt**. Người bị cấm: khách `2` và **tài xế `11`**.

| id | request_at | client (cấm?) | driver (cấm?) | status | Quyết định | `stats` sau bước này |
|---:|---|---|---|---|---|---|
| 1 | 2013-09-30 | 1 (No) | 10 (No) | cancelled_by_client | bỏ: **ngoài khoảng ngày** | `{}` |
| 2 | 2013-10-01 | 1 (No) | 10 (No) | completed | tính, hoàn thành | `{'01': [1, 0]}` |
| 3 | 2013-10-01 | 2 (**Yes**) | 12 (No) | cancelled_by_driver | bỏ: **khách bị cấm** | `{'01': [1, 0]}` |
| 4 | 2013-10-01 | 3 (No) | 11 (**Yes**) | cancelled_by_client | bỏ: **tài xế bị cấm** | `{'01': [1, 0]}` |
| 5 | 2013-10-01 | 4 (No) | 13 (No) | cancelled_by_driver | tính, **hủy** | `{'01': [2, 1]}` |
| 6 | 2013-10-01 | 3 (No) | 12 (No) | completed | tính, hoàn thành | `{'01': [3, 1]}` |
| 7 | 2013-10-02 | 2 (**Yes**) | 10 (No) | completed | bỏ: khách bị cấm | `{'01': [3, 1]}` |
| 8 | 2013-10-02 | 1 (No) | 11 (**Yes**) | completed | bỏ: tài xế bị cấm | `{'01': [3, 1]}` |
| 9 | 2013-10-03 | 1 (No) | 12 (No) | cancelled_by_client | tính, **hủy** | `{'01': [3, 1], '03': [1, 1]}` |
| 10 | 2013-10-03 | 4 (No) | 13 (No) | cancelled_by_driver | tính, **hủy** | `{'01': [3, 1], '03': [2, 2]}` |
| 11 | 2013-10-03 | 3 (No) | 10 (No) | completed | tính, hoàn thành | `{'01': [3, 1], '03': [3, 2]}` |
| 12 | 2013-10-03 | 4 (No) | 11 (**Yes**) | cancelled_by_driver | bỏ: tài xế bị cấm | `{'01': [3, 1], '03': [3, 2]}` |
| 13 | 2013-10-04 | 1 (No) | 10 (No) | cancelled_by_client | bỏ: **ngoài khoảng ngày** | `{'01': [3, 1], '03': [3, 2]}` |

Kết quả: ngày `01` có `1/3 = 0.33`; ngày `03` có `2/3 = 0.67`; ngày `02` **không xuất hiện** vì cả hai chuyến của ngày đó đều dính người bị cấm.

Hàm `trace_python` dưới đây in đúng bảng này để bạn đối chiếu (để in gọn, khóa của `stats` chỉ ghi 2 chữ số cuối của ngày: `'01'` là `2013-10-01`).

In [16]:
USERS_HARD = [
    (1, "No", "client"), (2, "Yes", "client"), (3, "No", "client"), (4, "No", "client"),
    (10, "No", "driver"), (11, "Yes", "driver"), (12, "No", "driver"), (13, "No", "driver"),
]
TRIPS_HARD = [
    (1, 1, 10, 1, "cancelled_by_client", "2013-09-30"),    # ngoài khoảng (trước)
    (2, 1, 10, 1, "completed", "2013-10-01"),
    (3, 2, 12, 1, "cancelled_by_driver", "2013-10-01"),    # khách 2 bị cấm
    (4, 3, 11, 6, "cancelled_by_client", "2013-10-01"),    # tài xế 11 bị cấm
    (5, 4, 13, 6, "cancelled_by_driver", "2013-10-01"),
    (6, 3, 12, 6, "completed", "2013-10-01"),
    (7, 2, 10, 1, "completed", "2013-10-02"),              # khách 2 bị cấm
    (8, 1, 11, 1, "completed", "2013-10-02"),              # tài xế 11 bị cấm
    (9, 1, 12, 12, "cancelled_by_client", "2013-10-03"),
    (10, 4, 13, 12, "cancelled_by_driver", "2013-10-03"),
    (11, 3, 10, 6, "completed", "2013-10-03"),
    (12, 4, 11, 12, "cancelled_by_driver", "2013-10-03"),  # tài xế 11 bị cấm
    (13, 1, 10, 1, "cancelled_by_client", "2013-10-04"),   # ngoài khoảng (sau)
]
EXPECTED_HARD = [("2013-10-01", 0.33), ("2013-10-03", 0.67)]

conn_hard = build_db(TRIPS_HARD, USERS_HARD)
print(trips_and_users_python(TRIPS_HARD, USERS_HARD))
assert trips_and_users_python(TRIPS_HARD, USERS_HARD) == EXPECTED_HARD

[('2013-10-01', 0.33), ('2013-10-03', 0.67)]


In [17]:
def trace_python(trips, users, start="2013-10-01", end="2013-10-03"):
    banned_ids = set()
    for users_id, banned, role in users:
        if banned == "Yes":
            banned_ids.add(users_id)
    print("Người bị cấm:", sorted(banned_ids))

    stats = {}
    for trip_id, client_id, driver_id, city_id, status, day in trips:
        if not (start <= day <= end):
            action = "bỏ: ngoài khoảng ngày"
        elif client_id in banned_ids:
            action = "bỏ: khách bị cấm"
        elif driver_id in banned_ids:
            action = "bỏ: tài xế bị cấm"
        else:
            if day not in stats:
                stats[day] = [0, 0]
            stats[day][0] += 1
            if status != "completed":
                stats[day][1] += 1
                action = "TÍNH, hủy"
            else:
                action = "TÍNH, hoàn thành"
        short = {d[-2:]: v for d, v in stats.items()}       # in gọn: chỉ 2 số cuối của ngày
        print(f"id={trip_id:>2} {day} c={client_id} d={driver_id} -> {action:<21} stats={short}")

    for day in sorted(stats):
        total, cancelled = stats[day]
        print(f"=> {day}: {cancelled}/{total} = {round_rate(cancelled, total)}")


trace_python(TRIPS_HARD, USERS_HARD)

Người bị cấm: [2, 11]
id= 1 2013-09-30 c=1 d=10 -> bỏ: ngoài khoảng ngày stats={}
id= 2 2013-10-01 c=1 d=10 -> TÍNH, hoàn thành      stats={'01': [1, 0]}
id= 3 2013-10-01 c=2 d=12 -> bỏ: khách bị cấm      stats={'01': [1, 0]}
id= 4 2013-10-01 c=3 d=11 -> bỏ: tài xế bị cấm     stats={'01': [1, 0]}
id= 5 2013-10-01 c=4 d=13 -> TÍNH, hủy             stats={'01': [2, 1]}
id= 6 2013-10-01 c=3 d=12 -> TÍNH, hoàn thành      stats={'01': [3, 1]}
id= 7 2013-10-02 c=2 d=10 -> bỏ: khách bị cấm      stats={'01': [3, 1]}
id= 8 2013-10-02 c=1 d=11 -> bỏ: tài xế bị cấm     stats={'01': [3, 1]}
id= 9 2013-10-03 c=1 d=12 -> TÍNH, hủy             stats={'01': [3, 1], '03': [1, 1]}
id=10 2013-10-03 c=4 d=13 -> TÍNH, hủy             stats={'01': [3, 1], '03': [2, 2]}
id=11 2013-10-03 c=3 d=10 -> TÍNH, hoàn thành      stats={'01': [3, 1], '03': [3, 2]}
id=12 2013-10-03 c=4 d=11 -> bỏ: tài xế bị cấm     stats={'01': [3, 1], '03': [3, 2]}
id=13 2013-10-04 c=1 d=10 -> bỏ: ngoài khoảng ngày stats={'01': [3, 1]

### Cách 1 vướng ở đâu? Và bước đột phá: dịch "bản thiết kế" sang SQL

Lời giải Python **đúng**, chạy `O(T + U)` thời gian (`T` = số chuyến, `U` = số người dùng; tra `set` trung bình `O(1)`), bộ nhớ `O(U + D)` (`D` = số ngày trong kết quả). Nhưng:

- LeetCode bài Database **chấm câu SQL** (hoặc Pandas), không chạy hàm Python thuần.
- Ngoài đời, dữ liệu nằm trong cơ sở dữ liệu có thể hàng trăm triệu hàng. Kéo hết về Python để lặp là chậm và tốn băng thông. Việc lọc và gom nhóm nên để **cơ sở dữ liệu tự làm**, vì nó có chỉ mục (index) và bộ tối ưu truy vấn.

Điều quý nhất Cách 1 cho ta là **bản thiết kế**: mỗi bước Python ứng với đúng một mệnh đề SQL.

| Bước trong Python | Mệnh đề SQL tương ứng |
|---|---|
| `banned_ids` + kiểm tra `client_id in banned_ids` | `JOIN Users` rồi `WHERE c.banned = 'No'` (Cách 2) **hoặc** `NOT IN (subquery)` (Cách 3) |
| kiểm tra cả `driver_id` | JOIN `Users` **lần thứ hai** / `NOT IN` thứ hai |
| `if not (start <= request_at <= end): continue` | `WHERE request_at BETWEEN '2013-10-01' AND '2013-10-03'` |
| `stats[request_at]` (gom theo ngày) | `GROUP BY request_at` |
| `stats[day][0] += 1` | `COUNT(*)` |
| `if status != "completed": stats[day][1] += 1` | `SUM(CASE WHEN status != 'completed' THEN 1 ELSE 0 END)` |
| `cancelled / total` rồi làm tròn | `ROUND(1.0 * SUM(...) / COUNT(*), 2)` hoặc `ROUND(AVG(status != 'completed'), 2)` |

SQL là ngôn ngữ **khai báo (declarative)**: ta mô tả **muốn kết quả gì** (lọc gì, nhóm theo gì, tính gì), còn **làm thế nào** (duyệt thứ tự nào, dùng chỉ mục nào) thì cơ sở dữ liệu tự quyết. Ta chỉ cần dịch đúng từng dòng của bảng trên.

## 10. Cách 2: JOIN `Users` hai lần + GROUP BY + `SUM(CASE)/COUNT` (cách chính)

Ta xây câu truy vấn **từng bước**, chạy thử sau mỗi bước, giống như lắp từng khúc ống nước rồi mở vòi kiểm tra.

### Bước 1: ghép hai lần và lọc ra các chuyến hợp lệ

```sql
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id     -- c = khách
JOIN Users AS d ON t.driver_id = d.users_id     -- d = tài xế
WHERE c.banned = 'No'                           -- khách không bị cấm
  AND d.banned = 'No'                           -- tài xế không bị cấm
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
```

Trên Example 1, ta mong đợi **7** chuyến hợp lệ: bỏ ba chuyến 2, 6, 8 của khách số 2.

In [18]:
show(conn_ex1, """
SELECT t.id, t.request_at, t.status, c.banned AS client_banned, d.banned AS driver_banned
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id
JOIN Users AS d ON t.driver_id = d.users_id
WHERE c.banned = 'No'
  AND d.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
ORDER BY t.id
""")

+----+------------+---------------------+---------------+---------------+
| id | request_at | status              | client_banned | driver_banned |
+----+------------+---------------------+---------------+---------------+
| 1  | 2013-10-01 | completed           | No            | No            |
| 3  | 2013-10-01 | completed           | No            | No            |
| 4  | 2013-10-01 | cancelled_by_client | No            | No            |
| 5  | 2013-10-02 | completed           | No            | No            |
| 7  | 2013-10-02 | completed           | No            | No            |
| 9  | 2013-10-03 | completed           | No            | No            |
| 10 | 2013-10-03 | cancelled_by_driver | No            | No            |
+----+------------+---------------------+---------------+---------------+ (7 hàng)


### Bước 2: gom theo ngày, đếm tổng và đếm số hủy

Giữ nguyên `FROM ... WHERE ...`, thêm `GROUP BY t.request_at`, và thay phần `SELECT` bằng hai hàm gộp:

- `COUNT(*)`: tổng số chuyến hợp lệ trong ngày (mẫu số).
- `SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END)`: số chuyến hợp lệ bị hủy (tử số).

Mong đợi trên Example 1: ngày `01` có `3` chuyến, `1` hủy; ngày `02` có `2`, `0` hủy; ngày `03` có `2`, `1` hủy.

In [19]:
show(conn_ex1, """
SELECT t.request_at,
       COUNT(*)                                                 AS total,
       SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) AS cancelled
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id
JOIN Users AS d ON t.driver_id = d.users_id
WHERE c.banned = 'No'
  AND d.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at
""")

+------------+-------+-----------+
| request_at | total | cancelled |
+------------+-------+-----------+
| 2013-10-01 | 3     | 1         |
| 2013-10-02 | 2     | 0         |
| 2013-10-03 | 2     | 1         |
+------------+-------+-----------+ (3 hàng)


### Bước 3: chia (ra số thực!), làm tròn, đặt đúng tên cột

```sql
ROUND(1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS `Cancellation Rate`
```

- `1.0 *` đứng **đầu** để phép chia ra số thực trên SQLite (MySQL thì sao cũng được, như đã học ở mục 8).
- `ROUND(…, 2)` bọc **bên ngoài** phép chia.
- Tên cột `Cancellation Rate` có **dấu cách**, nên bắt buộc phải **quote** (bọc tên). MySQL dùng dấu backtick `` ` ``. SQLite chấp nhận cả backtick, dấu nháy kép `"…"` lẫn ngoặc vuông `[…]`. Dùng **backtick** thì chạy được ở cả hai nơi. Ta quote luôn cả `Day` cho đồng bộ (không bắt buộc).

Ghép lại, ta có lời giải hoàn chỉnh `SQL_JOIN`:

In [20]:
SQL_JOIN = """
SELECT
    t.request_at AS `Day`,
    ROUND(
        1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS `Cancellation Rate`
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id
JOIN Users AS d ON t.driver_id = d.users_id
WHERE c.banned = 'No'
  AND d.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at
"""

show(conn_ex1, SQL_JOIN)
assert normalize(run_query(conn_ex1, SQL_JOIN)[1]) == EXPECTED_EX1
assert normalize(run_query(conn_hard, SQL_JOIN)[1]) == EXPECTED_HARD
print("SQL_JOIN đúng trên Example 1 và bộ dữ liệu khó.")

+------------+-------------------+
| Day        | Cancellation Rate |
+------------+-------------------+
| 2013-10-01 | 0.33              |
| 2013-10-02 | 0.0               |
| 2013-10-03 | 0.5               |
+------------+-------------------+ (3 hàng)
SQL_JOIN đúng trên Example 1 và bộ dữ liệu khó.


### Trace "đường ống" SQL trên bộ dữ liệu khó

SQL không có vòng lặp để `print` từng bước. Nhưng ta vẫn **trace** được bằng cách đếm số hàng còn lại **sau mỗi mệnh đề**, theo đúng thứ tự logic `FROM → JOIN → WHERE → GROUP BY → SELECT`:

| Giai đoạn | Số hàng còn lại | Hàng bị loại ở bước này |
|---|---:|---|
| `FROM Trips` | 13 | |
| sau 2 lần `JOIN` | 13 | không có (khóa ngoại luôn khớp đúng 1 người) |
| `WHERE c.banned = 'No'` | 11 | 3, 7 (khách 2) |
| `AND d.banned = 'No'` | 8 | 4, 8, 12 (tài xế 11) |
| `AND request_at BETWEEN …` | 6 | 1 (ngày 09-30), 13 (ngày 10-04) |
| `GROUP BY request_at` | 2 nhóm | ngày `02` không còn hàng nào nên không thành nhóm |

Rồi trong từng nhóm: ngày `01` gồm chuyến `2, 5, 6` có 1 hủy / 3; ngày `03` gồm chuyến `9, 10, 11` có 2 hủy / 3. So với bảng trace của Cách 1: **khớp từng chuyến**.

In [21]:
def trace_sql_pipeline(conn):
    base = """FROM Trips AS t
              JOIN Users AS c ON t.client_id = c.users_id
              JOIN Users AS d ON t.driver_id = d.users_id"""
    steps = [
        ("FROM Trips", "SELECT COUNT(*) FROM Trips"),
        ("sau 2 lần JOIN", f"SELECT COUNT(*) {base}"),
        ("+ khách không bị cấm", f"SELECT COUNT(*) {base} WHERE c.banned = 'No'"),
        ("+ tài xế không bị cấm", f"SELECT COUNT(*) {base} WHERE c.banned = 'No' AND d.banned = 'No'"),
        ("+ trong khoảng ngày", f"""SELECT COUNT(*) {base} WHERE c.banned = 'No' AND d.banned = 'No'
                                    AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'"""),
    ]
    for label, sql in steps:
        print(f"{label:<24}: {conn.execute(sql).fetchone()[0]:>2} hàng")

    print("GROUP BY request_at, từng nhóm:")
    group_sql = f"""
        SELECT t.request_at, GROUP_CONCAT(t.id), COUNT(*),
               SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END),
               ROUND(1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2)
        {base}
        WHERE c.banned = 'No' AND d.banned = 'No'
          AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
        GROUP BY t.request_at"""
    for day, ids, total, cancelled, rate in conn.execute(group_sql):
        print(f"  {day}: chuyến [{ids}] -> {cancelled} hủy / {total} tổng -> ROUND = {rate}")


trace_sql_pipeline(conn_hard)

FROM Trips              : 13 hàng
sau 2 lần JOIN          : 13 hàng
+ khách không bị cấm    : 11 hàng
+ tài xế không bị cấm   :  8 hàng
+ trong khoảng ngày     :  6 hàng
GROUP BY request_at, từng nhóm:
  2013-10-01: chuyến [2,5,6] -> 1 hủy / 3 tổng -> ROUND = 0.33
  2013-10-03: chuyến [9,10,11] -> 2 hủy / 3 tổng -> ROUND = 0.67


## 11. Cách 3: subquery `NOT IN`, lọc theo danh sách người bị cấm

**Cách 2 vướng ở đâu?** Nó đúng và là cách "chuẩn sách giáo khoa", nhưng hơi dài: hai lần JOIN, phải nghĩ alias `c`, `d`. Trong khi điều ta cần biết về `Users` chỉ là **"ai bị cấm"**, một danh sách thường rất ngắn.

**Mấu chốt cải tiến:** thay vì ghép cả bảng `Users` vào từng chuyến, ta **lập danh sách người bị cấm một lần** bằng subquery, rồi lọc chuyến bằng `NOT IN`. Đây chính là `banned_ids` của Cách 1 dịch thẳng sang SQL:

```sql
WHERE client_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND driver_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
```

Không còn JOIN, nên cũng không cần alias: mọi cột đều của `Trips`.

**Một biến thể tương đương:** `client_id IN (SELECT users_id FROM Users WHERE banned = 'No')`, tức "giữ nếu nằm trong danh sách người **không** bị cấm". Hai cách chỉ khác nhau khi dữ liệu **lỗi**: nếu một chuyến có `client_id` không tồn tại trong `Users`, thì `NOT IN (bị cấm)` **giữ** chuyến đó, còn `IN (không bị cấm)` và INNER JOIN **bỏ** nó. Đề đảm bảo khóa ngoại hợp lệ nên cả ba cho cùng kết quả.

In [22]:
SQL_NOT_IN = """
SELECT
    request_at AS `Day`,
    ROUND(1.0 * SUM(CASE WHEN status != 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS `Cancellation Rate`
FROM Trips
WHERE client_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND driver_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY request_at
"""

show(conn_hard, SQL_NOT_IN)
assert normalize(run_query(conn_ex1, SQL_NOT_IN)[1]) == EXPECTED_EX1
assert normalize(run_query(conn_hard, SQL_NOT_IN)[1]) == EXPECTED_HARD
print("SQL_NOT_IN đúng trên cả hai bộ dữ liệu.")

+------------+-------------------+
| Day        | Cancellation Rate |
+------------+-------------------+
| 2013-10-01 | 0.33              |
| 2013-10-03 | 0.67              |
+------------+-------------------+ (2 hàng)
SQL_NOT_IN đúng trên cả hai bộ dữ liệu.


## 12. Cách 4: `AVG(status != 'completed')`, kiểu MySQL

**Cách 3 còn dài ở đâu?** Ở công thức tỉ lệ: `1.0 * SUM(CASE WHEN ... THEN 1 ELSE 0 END) / COUNT(*)`. Ta đang tự tay làm phép "tổng các số 0/1 chia số hàng", mà đó chính là **định nghĩa của trung bình**!

**Mấu chốt:** tỉ lệ hủy = **trung bình** của chỉ báo 0/1 "chuyến này có bị hủy không". Trong MySQL, `status != 'completed'` đã là 0/1, nên:

```sql
ROUND(AVG(status != 'completed'), 2) AS `Cancellation Rate`
```

**Trên SQLite thì sao?** Chạy **y hệt**: phép so sánh cũng trả 0/1, và `AVG` của SQLite luôn trả số thực, nên **không** dính chia nguyên. Đây là lý do cách này được ưa chuộng: ngắn và không có bẫy phép chia.

Các cách viết tương đương (chọn cái bạn thấy dễ đọc nhất):

| Cách viết | MySQL | SQLite |
|---|:---:|:---:|
| `AVG(status != 'completed')` | chạy | chạy |
| `AVG(CASE WHEN status != 'completed' THEN 1.0 ELSE 0 END)` | chạy | chạy (chuẩn SQL, chạy cả PostgreSQL) |
| `AVG(IF(status != 'completed', 1, 0))` | chạy | `IF` chỉ có từ SQLite 3.48; bản cũ hơn dùng `IIF` (có từ 3.32) |
| `SUM(status != 'completed') / COUNT(*)` | chạy | **ra 0** (chia nguyên!) |
| `AVG(status LIKE 'cancelled%')` | chạy | chạy (`LIKE` so khớp mẫu; `%` = chuỗi bất kỳ) |

**Ghi chú nâng cao:** `AVG` **bỏ qua** giá trị `NULL`, còn `COUNT(*)` thì không. Nếu `status` có thể là `NULL` thì hai công thức sẽ khác nhau. Đề đảm bảo `status` luôn thuộc 3 giá trị nên không sao.

Ta ghép `AVG` với bộ lọc `NOT IN` của Cách 3 để được lời giải **ngắn nhất**. (Ghép `AVG` với JOIN hai lần cũng được; đó là Bài tập 4.)

In [23]:
SQL_AVG = """
SELECT
    request_at AS `Day`,
    ROUND(AVG(status != 'completed'), 2) AS `Cancellation Rate`
FROM Trips
WHERE client_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND driver_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY request_at
"""

show(conn_ex1, SQL_AVG)

# Cách viết chuẩn SQL với CASE ... 1.0 ... cho cùng kết quả
SQL_AVG_CASE = SQL_AVG.replace("AVG(status != 'completed')",
                               "AVG(CASE WHEN status != 'completed' THEN 1.0 ELSE 0 END)")
for conn, expected in [(conn_ex1, EXPECTED_EX1), (conn_hard, EXPECTED_HARD)]:
    assert normalize(run_query(conn, SQL_AVG)[1]) == expected
    assert normalize(run_query(conn, SQL_AVG_CASE)[1]) == expected
print("SQL_AVG và SQL_AVG_CASE đúng trên cả hai bộ dữ liệu.")

+------------+-------------------+
| Day        | Cancellation Rate |
+------------+-------------------+
| 2013-10-01 | 0.33              |
| 2013-10-02 | 0.0               |
| 2013-10-03 | 0.5               |
+------------+-------------------+ (3 hàng)
SQL_AVG và SQL_AVG_CASE đúng trên cả hai bộ dữ liệu.


## 13. Cách 5: Pandas (tùy chọn)

LeetCode cũng cho nộp bằng **Pandas**, thư viện xử lý bảng phổ biến nhất của Python (không thuộc thư viện chuẩn; cài bằng `pip install pandas`). Nếu máy bạn chưa có pandas, các cell dưới tự bỏ qua.

Vài khái niệm nền:

- **`DataFrame`** = một **bảng** (giống bảng SQL). **`Series`** = một **cột**.
- `df["status"] != "completed"` tạo một `Series` toàn `True`/`False`, **mỗi hàng một giá trị**. Ta gọi nó là **mặt nạ (boolean mask)**.
- `df[mask]` giữ lại các hàng có mặt nạ `True`, tương đương `WHERE`.
- Nối nhiều điều kiện bằng `&` (và), `|` (hoặc), `~` (phủ định), **mỗi điều kiện bọc trong ngoặc**. **Không** dùng `and`/`or`/`not` của Python.
- `s.isin(danh_sach)` tương đương `IN`; `~s.isin(...)` tương đương `NOT IN`; `s.between(a, b)` tương đương `BETWEEN` (lấy cả hai đầu).
- `df.groupby("cot")["x"].mean()` tương đương `GROUP BY cot` với `AVG(x)`. Trung bình của cột `True`/`False` chính là **tỉ lệ `True`**, giống Cách 4.

Thử trên một bảng mini:

In [24]:
try:
    import pandas as pd
    HAS_PANDAS = True
except ImportError:
    HAS_PANDAS = False
    print("Chưa cài pandas: bỏ qua Cách 5 (cài bằng: pip install pandas).")

if HAS_PANDAS:
    df = pd.DataFrame({"day": ["d1", "d1", "d2"],
                       "status": ["completed", "cancelled_by_client", "completed"]})
    mask = df["status"] != "completed"
    print("Mặt nạ:", mask.tolist())
    print("Các hàng có mặt nạ True:")
    print(df[mask])
    print("Tỉ lệ hủy theo ngày (mean của True/False):")
    print(df.assign(is_cancelled=mask).groupby("day")["is_cancelled"].mean())

Mặt nạ: [False, True, False]
Các hàng có mặt nạ True:
  day               status
1  d1  cancelled_by_client
Tỉ lệ hủy theo ngày (mean của True/False):
day
d1    0.5
d2    0.0
Name: is_cancelled, dtype: float64


In [25]:
if HAS_PANDAS:
    # Trên LeetCode bạn chỉ dán phần "import pandas as pd" và hàm bên dưới.
    def trips_and_users(trips: pd.DataFrame, users: pd.DataFrame) -> pd.DataFrame:
        banned_ids = users.loc[users["banned"] == "Yes", "users_id"]
        valid = trips[
            (~trips["client_id"].isin(banned_ids))
            & (~trips["driver_id"].isin(banned_ids))
            & (trips["request_at"].between("2013-10-01", "2013-10-03"))
        ]
        result = (
            valid.assign(is_cancelled=valid["status"] != "completed")
            .groupby("request_at", as_index=False)["is_cancelled"]
            .mean()
        )
        result["is_cancelled"] = result["is_cancelled"].round(2)
        return result.rename(columns={"request_at": "Day", "is_cancelled": "Cancellation Rate"})

    def to_frames(trips_rows, users_rows):
        """Dựng DataFrame giống cách LeetCode dựng input (cột id dùng kiểu Int64)."""
        trips = pd.DataFrame(trips_rows, columns=["id", "client_id", "driver_id", "city_id",
                                                  "status", "request_at"])
        trips = trips.astype({"id": "Int64", "client_id": "Int64", "driver_id": "Int64",
                              "city_id": "Int64", "status": "object", "request_at": "object"})
        users = pd.DataFrame(users_rows, columns=["users_id", "banned", "role"])
        users = users.astype({"users_id": "Int64", "banned": "object", "role": "object"})
        return trips, users

    out = trips_and_users(*to_frames(TRIPS_EX1, USERS_EX1))
    print(out)
    assert normalize(zip(out["Day"], out["Cancellation Rate"])) == EXPECTED_EX1
    out_hard = trips_and_users(*to_frames(TRIPS_HARD, USERS_HARD))
    assert normalize(zip(out_hard["Day"], out_hard["Cancellation Rate"])) == EXPECTED_HARD
    print("Pandas đúng trên cả hai bộ dữ liệu.")

          Day  Cancellation Rate
0  2013-10-01               0.33
1  2013-10-02               0.00
2  2013-10-03               0.50
Pandas đúng trên cả hai bộ dữ liệu.


Hai lưu ý về bản Pandas:

- `*to_frames(...)` "mở" tuple `(trips, users)` thành hai đối số, tức là `trips_and_users(trips, users)`.
- `.round(2)` của Pandas làm tròn "nửa về số chẵn" giống `round` của Python (ở ca đúng nửa như `0.125` sẽ ra `0.12`, còn MySQL ra `0.13`). Nếu muốn khớp MySQL tuyệt đối, hãy đếm `total` và `cancelled` theo nhóm rồi áp công thức số nguyên như `round_rate`. Các test trong notebook này không có ca đúng nửa.

## 14. Code nộp LeetCode (MySQL)

Trên trang bài, chọn ngôn ngữ **MySQL** rồi dán **một** trong hai query sau. Bản chính (Cách 2) dễ đọc, chạy được ở mọi hệ quản trị:

```sql
SELECT
    t.request_at AS `Day`,
    ROUND(
        1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS `Cancellation Rate`
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id
JOIN Users AS d ON t.driver_id = d.users_id
WHERE c.banned = 'No'
  AND d.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at;
```

Bản ngắn (Cách 3 + 4):

```sql
SELECT request_at AS `Day`,
       ROUND(AVG(status != 'completed'), 2) AS `Cancellation Rate`
FROM Trips
WHERE client_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND driver_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY request_at;
```

Cả hai đều là cú pháp MySQL hợp lệ **và** chạy được nguyên văn trên SQLite (ta kiểm ngay dưới đây). Khác biệt duy nhất bạn sẽ thấy: MySQL in `0.00`, `0.50` (số thập phân giữ 2 chữ số), SQLite in `0.0`, `0.5` (số thực). Giá trị như nhau nên LeetCode vẫn chấm đúng. Dấu `;` ở cuối là tùy chọn.

In [26]:
LEETCODE_MYSQL = """
SELECT
    t.request_at AS `Day`,
    ROUND(
        1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*),
        2
    ) AS `Cancellation Rate`
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id
JOIN Users AS d ON t.driver_id = d.users_id
WHERE c.banned = 'No'
  AND d.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at;
"""

LEETCODE_MYSQL_SHORT = """
SELECT request_at AS `Day`,
       ROUND(AVG(status != 'completed'), 2) AS `Cancellation Rate`
FROM Trips
WHERE client_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND driver_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY request_at;
"""


def check_day_query(sql):
    """Chạy một query 'tỉ lệ hủy theo ngày' trên 2 bộ dữ liệu mẫu, so với kết quả mong đợi."""
    for name, trips, users, expected in [("Example 1", TRIPS_EX1, USERS_EX1, EXPECTED_EX1),
                                         ("bộ khó", TRIPS_HARD, USERS_HARD, EXPECTED_HARD)]:
        got = normalize(run_query(build_db(trips, users), sql)[1])
        assert got == expected, (name, got, expected)
    print("PASS: Example 1 + bộ dữ liệu khó")


show(conn_ex1, LEETCODE_MYSQL)
check_day_query(LEETCODE_MYSQL)
check_day_query(LEETCODE_MYSQL_SHORT)

+------------+-------------------+
| Day        | Cancellation Rate |
+------------+-------------------+
| 2013-10-01 | 0.33              |
| 2013-10-02 | 0.0               |
| 2013-10-03 | 0.5               |
+------------+-------------------+ (3 hàng)
PASS: Example 1 + bộ dữ liệu khó
PASS: Example 1 + bộ dữ liệu khó


### Bộ test edge case và đối chiếu chéo mọi cách

Ta gom **mọi lời giải** vào một dict `SOLVERS`: mỗi lời giải là một hàm nhận `(trips_rows, users_rows)` và trả về kết quả đã `normalize`. Với các query SQL, hàm `via_sql(sql)` tạo cơ sở dữ liệu mới, chạy query, rồi chuẩn hóa kết quả.

Các ca kiểm thử, mỗi ca nhắm vào một cái bẫy:

1. **Example 1** của đề.
2. **Bộ khó**: tài xế bị cấm, chuyến ngoài khoảng ngày, ngày toàn chuyến bị cấm.
3. **Bảng `Trips` rỗng**: kết quả rỗng, không lỗi chia cho 0.
4. **Mọi chuyến đều dính người bị cấm**: kết quả rỗng.
5. **Một ngày hủy 100%**: ra `1.0`.
6. **Cả khách lẫn tài xế cùng bị cấm** trong một chuyến: chuyến đó bị bỏ **một lần**, không làm hỏng đếm.
7. **Đúng hai đầu mút** `2013-10-01` và `2013-10-03` được tính; `2013-09-30` và `2013-10-04` thì không.

Sau đó là **đối chiếu chéo ngẫu nhiên**: sinh 300 bộ dữ liệu nhỏ (`random.seed` cố định để chạy lại ra y hệt), chạy tất cả các cách, yêu cầu **tất cả cho cùng một kết quả** với Cách 1 (Python thuần, dễ tin nhất). Mỗi ngày sinh tối đa 7 chuyến, nên tỉ lệ có mẫu số từ 1 đến 7 và **không bao giờ** rơi vào ca "đúng nửa" (ca đó chỉ có thể xảy ra khi mẫu số chia hết cho 8, ví dụ `1/8`, `3/40`), các cách làm tròn sẽ trùng nhau.

In [27]:
def via_sql(sql):
    def solve(trips_rows, users_rows):
        return normalize(run_query(build_db(trips_rows, users_rows), sql)[1])
    return solve


def via_python(trips_rows, users_rows):
    return normalize(trips_and_users_python(trips_rows, users_rows))


SOLVERS = {
    "python": via_python,
    "sql_join": via_sql(SQL_JOIN),
    "sql_not_in": via_sql(SQL_NOT_IN),
    "sql_avg": via_sql(SQL_AVG),
    "leetcode": via_sql(LEETCODE_MYSQL),
    "leetcode_short": via_sql(LEETCODE_MYSQL_SHORT),
}
if HAS_PANDAS:
    def via_pandas(trips_rows, users_rows):
        out = trips_and_users(*to_frames(trips_rows, users_rows))
        return normalize(zip(out["Day"], out["Cancellation Rate"]))
    SOLVERS["pandas"] = via_pandas

U = [(1, "No", "client"), (2, "Yes", "client"), (10, "No", "driver"), (11, "Yes", "driver")]
EDGE_CASES = [
    ("Example 1", TRIPS_EX1, USERS_EX1, EXPECTED_EX1),
    ("bộ khó", TRIPS_HARD, USERS_HARD, EXPECTED_HARD),
    ("Trips rỗng", [], USERS_EX1, []),
    ("mọi chuyến dính người bị cấm", [(1, 2, 10, 1, "completed", "2013-10-01"),
                                     (2, 1, 11, 1, "cancelled_by_client", "2013-10-02")], U, []),
    ("hủy 100%", [(1, 1, 10, 1, "cancelled_by_client", "2013-10-02"),
                  (2, 1, 10, 1, "cancelled_by_driver", "2013-10-02")], U, [("2013-10-02", 1.0)]),
    ("cả hai cùng bị cấm", [(1, 2, 11, 1, "cancelled_by_driver", "2013-10-03"),
                            (2, 1, 10, 1, "completed", "2013-10-03")], U, [("2013-10-03", 0.0)]),
    ("hai đầu mút", [(1, 1, 10, 1, "cancelled_by_client", "2013-09-30"),
                     (2, 1, 10, 1, "completed", "2013-10-01"),
                     (3, 1, 10, 1, "cancelled_by_client", "2013-10-03"),
                     (4, 1, 10, 1, "completed", "2013-10-04")], U,
     [("2013-10-01", 0.0), ("2013-10-03", 1.0)]),
]

for name, trips_rows, users_rows, expected in EDGE_CASES:
    for solver_name, solve in SOLVERS.items():
        got = solve(trips_rows, users_rows)
        assert got == expected, (name, solver_name, got, expected)
    print(f"PASS {name:<30} -> {expected}")
print("Các lời giải được kiểm:", list(SOLVERS))

PASS Example 1                      -> [('2013-10-01', 0.33), ('2013-10-02', 0.0), ('2013-10-03', 0.5)]
PASS bộ khó                         -> [('2013-10-01', 0.33), ('2013-10-03', 0.67)]
PASS Trips rỗng                     -> []
PASS mọi chuyến dính người bị cấm   -> []
PASS hủy 100%                       -> [('2013-10-02', 1.0)]
PASS cả hai cùng bị cấm             -> [('2013-10-03', 0.0)]
PASS hai đầu mút                    -> [('2013-10-01', 0.0), ('2013-10-03', 1.0)]
Các lời giải được kiểm: ['python', 'sql_join', 'sql_not_in', 'sql_avg', 'leetcode', 'leetcode_short', 'pandas']


In [28]:
import random

random.seed(262)
DAYS = ["2013-09-30", "2013-10-01", "2013-10-02", "2013-10-03", "2013-10-04"]
STATUSES = ["completed", "cancelled_by_driver", "cancelled_by_client"]


def random_case():
    users = []
    for uid in range(1, 5):
        users.append((uid, "Yes" if random.random() < 0.25 else "No", "client"))
    for uid in range(10, 14):
        users.append((uid, "Yes" if random.random() < 0.25 else "No", "driver"))
    trips, next_id = [], 1
    for day in DAYS:
        for _ in range(random.randint(0, 7)):             # tối đa 7 chuyến mỗi ngày
            trips.append((next_id, random.randint(1, 4), random.randint(10, 13),
                          random.randint(1, 3), random.choice(STATUSES), day))
            next_id += 1
    random.shuffle(trips)                                  # thứ tự hàng trong bảng không quan trọng
    return trips, users


non_empty = 0
for _ in range(300):
    trips_rows, users_rows = random_case()
    reference = via_python(trips_rows, users_rows)
    non_empty += bool(reference)
    for solver_name, solve in SOLVERS.items():
        assert solve(trips_rows, users_rows) == reference, (solver_name, trips_rows, users_rows)
print(f"300 bộ ngẫu nhiên: {len(SOLVERS)} lời giải luôn trùng nhau ({non_empty} bộ có kết quả khác rỗng).")

300 bộ ngẫu nhiên: 7 lời giải luôn trùng nhau (285 bộ có kết quả khác rỗng).


## 15. Vì sao đúng? Độ phức tạp? So sánh các cách

### Vì sao Cách 2 đúng (invariant bằng lời)

**Bất biến sau `WHERE`:** một chuyến còn lại **khi và chỉ khi** (a) ngày của nó nằm trong `[2013-10-01, 2013-10-03]`, (b) khách không bị cấm, (c) tài xế không bị cấm. Đó đúng là định nghĩa "chuyến hợp lệ" của đề.

- **JOIN không làm mất hay nhân bản chuyến:** `users_id` là khóa chính nên mỗi `client_id` khớp **đúng một** hàng `c`, mỗi `driver_id` khớp **đúng một** hàng `d`. Mỗi chuyến sinh ra đúng một hàng sau hai lần JOIN, mang theo đúng cờ `banned` của hai người.
- **Mỗi nhóm của `GROUP BY` là đúng tập chuyến hợp lệ của một ngày.** Do đó `COUNT(*)` là mẫu số đề yêu cầu.
- **Tử số:** `status` chỉ có 3 giá trị, nên "không `completed`" tương đương "bị hủy bởi khách hoặc tài xế". `SUM(CASE …)` đếm đúng các chuyến đó.
- **"Ít nhất một chuyến":** ngày không còn chuyến hợp lệ nào thì không có hàng nào vào nhóm, nên không có hàng kết quả. Vì vậy **không bao giờ chia cho 0**.
- `1.0 *` bảo đảm phép chia là chia thực; `ROUND(…, 2)` làm tròn đúng yêu cầu.

Cách 3 thay "JOIN rồi kiểm cờ" bằng "kiểm không thuộc danh sách bị cấm", cho cùng tập chuyến hợp lệ (với dữ liệu thỏa khóa ngoại). Cách 4 thay `SUM/COUNT` bằng `AVG` của chỉ báo 0/1, vốn là cùng một con số.

### Độ phức tạp

Gọi `T` = số hàng `Trips`, `U` = số hàng `Users`, `B` = số người bị cấm, `D` = số ngày trong kết quả (ở đây `D ≤ 3`). Với SQL, thời gian **phụ thuộc bộ tối ưu truy vấn** của từng hệ quản trị; các con số dưới đây là ước lượng điển hình.

- **Cách 1 (Python):** `O(U + T)` thời gian (tra `set` trung bình `O(1)`), cộng `O(D log D)` để sắp ngày. Bộ nhớ `O(B + D)`.
- **Cách 2 (JOIN×2):** mỗi lần JOIN tra khóa chính qua chỉ mục (B-tree) mất `O(log U)`, nên tổng `O(T log U)`. GROUP BY bằng sắp xếp là `O(T log T)`, bằng bảng băm là `O(T)`. Bộ nhớ phụ `O(D)` cho các nhóm (cộng bộ đệm tạm nếu phải sắp xếp).
- **Cách 3 / 4 (NOT IN):** subquery chạy **một lần**, ra `B` mã. Kiểm `NOT IN` qua chỉ mục tạm hoặc bảng băm, tổng khoảng `O(U + T log B)` hoặc `O(U + T)`. Bộ nhớ `O(B + D)`.
- **Cách 5 (Pandas):** `isin` dùng bảng băm `O(U + T)`; `groupby` `O(T)` cộng sắp khóa nhóm. Bộ nhớ `O(T)` vì tạo mặt nạ và bảng con `valid`.

### Bảng so sánh

| Cách | Lọc người bị cấm | Tính tỉ lệ | Ưu điểm | Nhược điểm |
|---|---|---|---|---|
| 1. Python thuần | `set` + `in` | `dict` đếm + `round_rate` | dễ hiểu nhất, làm "đáp án chuẩn" để test | không nộp được ở bài Database |
| 2. JOIN×2 + `SUM(CASE)/COUNT` | JOIN 2 lần + `WHERE banned = 'No'` | `1.0*SUM(CASE)/COUNT(*)` | chuẩn SQL, chạy mọi hệ quản trị, dễ mở rộng (cần thêm cột của `Users`) | dài, dễ quên lọc tài xế hoặc quên `1.0` |
| 3. `NOT IN` subquery | `NOT IN (SELECT … 'Yes')` | như Cách 2 | đọc giống đề, không cần alias | phải nhớ bẫy `NULL` của `NOT IN` |
| 4. `AVG(status != 'completed')` | thường đi với `NOT IN` | `AVG` của 0/1 | ngắn nhất, không có bẫy chia nguyên | dựa vào việc so sánh ra 0/1 (MySQL, SQLite); PostgreSQL cần `CASE` |
| 5. Pandas | `~isin` | `groupby().mean()` | quen với dân Python/data | không thuộc thư viện chuẩn; `.round` làm tròn nửa về số chẵn |

## 16. Lỗi người mới hay gặp

Gần như mọi lỗi dưới đây đều **chạy được**, không báo lỗi gì, chỉ ra **kết quả sai** (riêng Lỗi 6 là lỗi cú pháp). Đó là lý do chúng nguy hiểm. Ta chạy từng query sai trên **bộ dữ liệu khó** và so với đáp án đúng `[('2013-10-01', 0.33), ('2013-10-03', 0.67)]`.

### Lỗi 1: chỉ lọc khách, **quên lọc tài xế** bị cấm

```sql
FROM Trips t JOIN Users c ON t.client_id = c.users_id
WHERE c.banned = 'No' AND t.request_at BETWEEN ...
```

**Vì sao sai:** đề nói rõ *both client and driver must not be banned*. Example 1 không có tài xế bị cấm nên query sai này **vẫn qua Example 1**, rồi trượt test ẩn. Trên bộ khó, các chuyến 4, 8, 12 (tài xế 11) bị tính oan. Ngày `02` còn "sống lại" với tỉ lệ `0.0`.

**Sửa:** JOIN `Users` lần thứ hai cho tài xế và thêm `d.banned = 'No'` (hoặc thêm `driver_id NOT IN (...)`).

### Lỗi 2: chia nguyên trên SQLite

```sql
ROUND(SUM(CASE ...) / COUNT(*), 2)          -- SQLite: 1/3 = 0, 2/3 = 0
ROUND(SUM(CASE ...) / COUNT(*) * 1.0, 2)    -- nhân 1.0 SAU khi đã chia nguyên: vẫn 0
```

**Vì sao sai:** cả hai vế là số nguyên nên SQLite (và PostgreSQL, SQL Server) chia nguyên. Trên MySQL query này lại **đúng**, nên rất dễ bị "máy mình sai, LeetCode đúng" hoặc ngược lại khi đổi hệ quản trị.

**Sửa:** `1.0 * SUM(...) / COUNT(*)` (đặt `1.0` **trước** phép chia), hoặc dùng `AVG(...)`.

In [29]:
def compare_wrong(label, wrong_sql, conn=conn_hard, expected=EXPECTED_HARD):
    got = normalize(run_query(conn, wrong_sql)[1])
    print(label)
    print("   kết quả :", got)
    print("   đúng là :", expected, "->", "trùng đáp án" if got == expected else "SAI")


WRONG_NO_DRIVER = """
SELECT t.request_at AS `Day`,
       ROUND(1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS `Cancellation Rate`
FROM Trips t
JOIN Users c ON t.client_id = c.users_id
WHERE c.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at
"""
compare_wrong("Lỗi 1 trên Example 1 (lọt lưới!):", WRONG_NO_DRIVER, conn_ex1, EXPECTED_EX1)
compare_wrong("Lỗi 1 trên bộ khó:", WRONG_NO_DRIVER)

WRONG_INT_DIV = SQL_JOIN.replace("1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*)",
                                 "SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*)")
compare_wrong("Lỗi 2a: SUM/COUNT không có 1.0:", WRONG_INT_DIV)
WRONG_LATE_CAST = SQL_JOIN.replace("1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*)",
                                   "SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*) * 1.0")
compare_wrong("Lỗi 2b: nhân 1.0 sau khi chia:", WRONG_LATE_CAST)

Lỗi 1 trên Example 1 (lọt lưới!):
   kết quả : [('2013-10-01', 0.33), ('2013-10-02', 0.0), ('2013-10-03', 0.5)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-02', 0.0), ('2013-10-03', 0.5)] -> trùng đáp án
Lỗi 1 trên bộ khó:
   kết quả : [('2013-10-01', 0.5), ('2013-10-02', 0.0), ('2013-10-03', 0.75)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-03', 0.67)] -> SAI
Lỗi 2a: SUM/COUNT không có 1.0:
   kết quả : [('2013-10-01', 0.0), ('2013-10-03', 0.0)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-03', 0.67)] -> SAI
Lỗi 2b: nhân 1.0 sau khi chia:
   kết quả : [('2013-10-01', 0.0), ('2013-10-03', 0.0)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-03', 0.67)] -> SAI


### Lỗi 3: quên khoảng ngày

**Vì sao sai:** thiếu `AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'` thì mọi ngày trong bảng đều được in ra. Example 1 chỉ có đúng 3 ngày đó nên **lại lọt lưới**; bộ khó thì thừa ra ngày `09-30` và `10-04`.

**Sửa:** luôn đọc lại đề tìm các con số, ngày tháng cụ thể. Mỗi con số trong đề gần như chắc chắn là một điều kiện phải có trong `WHERE`.

### Lỗi 4: LEFT JOIN và đặt điều kiện cấm vào `ON`

```sql
FROM Trips t
LEFT JOIN Users c ON t.client_id = c.users_id AND c.banned = 'No'
LEFT JOIN Users d ON t.driver_id = d.users_id AND d.banned = 'No'
WHERE t.request_at BETWEEN ...
```

**Vì sao sai:** điều kiện trong `ON` chỉ quyết định **có ghép được** hay không. Với LEFT JOIN, chuyến **không ghép được vẫn được giữ lại** (các cột bên phải thành `NULL`). Kết quả: chuyến của người bị cấm **không bị loại**, mọi chuyến trong khoảng ngày đều bị đếm.

**Sửa:** dùng INNER JOIN (`JOIN`), hoặc giữ LEFT JOIN nhưng đưa điều kiện xuống `WHERE c.banned = 'No' AND d.banned = 'No'`. Khi đó hàng có `banned` là `NULL` không thỏa `= 'No'` nên bị loại, và LEFT JOIN cư xử như INNER JOIN. Quy tắc nhớ: **điều kiện để lọc hàng đặt ở `WHERE`; điều kiện để ghép đặt ở `ON`.**

In [30]:
WRONG_NO_DATE = SQL_JOIN.replace("AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'", "")
compare_wrong("Lỗi 3 trên Example 1 (lọt lưới!):", WRONG_NO_DATE, conn_ex1, EXPECTED_EX1)
compare_wrong("Lỗi 3 trên bộ khó:", WRONG_NO_DATE)

WRONG_LEFT_ON = """
SELECT t.request_at AS `Day`,
       ROUND(1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS `Cancellation Rate`
FROM Trips t
LEFT JOIN Users c ON t.client_id = c.users_id AND c.banned = 'No'
LEFT JOIN Users d ON t.driver_id = d.users_id AND d.banned = 'No'
WHERE t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at
"""
compare_wrong("Lỗi 4: LEFT JOIN, điều kiện cấm nằm trong ON:", WRONG_LEFT_ON)

FIXED_LEFT_WHERE = """
SELECT t.request_at AS `Day`,
       ROUND(1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS `Cancellation Rate`
FROM Trips t
LEFT JOIN Users c ON t.client_id = c.users_id
LEFT JOIN Users d ON t.driver_id = d.users_id
WHERE c.banned = 'No' AND d.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at
"""
compare_wrong("Sửa lỗi 4: LEFT JOIN nhưng lọc ở WHERE:", FIXED_LEFT_WHERE)

Lỗi 3 trên Example 1 (lọt lưới!):
   kết quả : [('2013-10-01', 0.33), ('2013-10-02', 0.0), ('2013-10-03', 0.5)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-02', 0.0), ('2013-10-03', 0.5)] -> trùng đáp án
Lỗi 3 trên bộ khó:
   kết quả : [('2013-09-30', 1.0), ('2013-10-01', 0.33), ('2013-10-03', 0.67), ('2013-10-04', 1.0)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-03', 0.67)] -> SAI
Lỗi 4: LEFT JOIN, điều kiện cấm nằm trong ON:
   kết quả : [('2013-10-01', 0.6), ('2013-10-02', 0.0), ('2013-10-03', 0.75)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-03', 0.67)] -> SAI
Sửa lỗi 4: LEFT JOIN nhưng lọc ở WHERE:
   kết quả : [('2013-10-01', 0.33), ('2013-10-03', 0.67)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-03', 0.67)] -> trùng đáp án


### Lỗi 5: JOIN **một lần** với `OR`

```sql
FROM Trips t JOIN Users u ON t.client_id = u.users_id OR t.driver_id = u.users_id
WHERE u.banned = 'No'
```

**Vì sao sai:** mỗi chuyến ghép được với **hai** hàng (khách và tài xế). `WHERE u.banned = 'No'` giữ lại hàng nào có người **không** bị cấm, nên chuyến có khách bị cấm vẫn lọt nhờ hàng của tài xế, còn chuyến hợp lệ bị **đếm hai lần**. Cả tử số lẫn mẫu số đều sai.

### Lỗi 6: tên cột có dấu cách mà không quote

`AS Cancellation Rate` gây **lỗi cú pháp**: SQL hiểu `Cancellation` là alias, rồi không biết `Rate` là gì. Phải viết ``AS `Cancellation Rate` `` (MySQL), hoặc `AS "Cancellation Rate"` (chuẩn SQL / SQLite).

### Lỗi 7: gõ sai trạng thái, hoặc chỉ đếm một loại hủy

Đề **viết** chữ "canceled" (một chữ `l`, kiểu Mỹ), nhưng **dữ liệu** là `'cancelled_by_driver'`, `'cancelled_by_client'` (hai chữ `l`). Viết `status = 'canceled'` hay `status = 'cancelled'` thì **không khớp hàng nào**: mọi tỉ lệ thành `0.0`, không có thông báo lỗi nào. Tương tự, chỉ đếm `'cancelled_by_driver'` là quên mất khách hủy. **An toàn nhất:** `status != 'completed'`, hoặc liệt kê đủ `status IN ('cancelled_by_driver', 'cancelled_by_client')`.

### Lỗi 8: làm tròn sai chỗ

`ROUND(SUM(...), 2) / COUNT(*)` làm tròn **tử số** (vốn là số nguyên, làm tròn chẳng thay đổi gì) rồi mới chia, nên kết quả **không được làm tròn**: `0.3333…` thay vì `0.33`. `ROUND` phải bọc **toàn bộ** phép chia.

In [31]:
WRONG_OR_JOIN = """
SELECT t.request_at AS `Day`,
       ROUND(1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2) AS `Cancellation Rate`
FROM Trips t
JOIN Users u ON t.client_id = u.users_id OR t.driver_id = u.users_id
WHERE u.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at
"""
print("Lỗi 5: số hàng sau JOIN với OR =",
      conn_hard.execute("SELECT COUNT(*) FROM Trips t JOIN Users u "
                        "ON t.client_id = u.users_id OR t.driver_id = u.users_id").fetchone()[0],
      "(gấp đôi 13 chuyến)")
compare_wrong("Lỗi 5: JOIN một lần với OR:", WRONG_OR_JOIN)

try:
    conn_hard.execute("SELECT request_at AS Cancellation Rate FROM Trips")
except sqlite3.OperationalError as error:
    print("\nLỗi 6: alias có dấu cách không quote ->", error)

WRONG_TYPO = SQL_JOIN.replace("t.status != 'completed'", "t.status = 'canceled'")
compare_wrong("\nLỗi 7: gõ 'canceled' (sai chính tả so với dữ liệu):", WRONG_TYPO)

WRONG_ROUND = SQL_JOIN.replace(
    "ROUND(\n        1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*),\n        2\n    )",
    "ROUND(1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END), 2) / COUNT(*)")
print("\nLỗi 8: ROUND chỉ bọc tử số ->", run_query(conn_hard, WRONG_ROUND)[1])

Lỗi 5: số hàng sau JOIN với OR = 26 (gấp đôi 13 chuyến)
Lỗi 5: JOIN một lần với OR:
   kết quả : [('2013-10-01', 0.5), ('2013-10-02', 0.0), ('2013-10-03', 0.71)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-03', 0.67)] -> SAI

Lỗi 6: alias có dấu cách không quote -> near "Rate": syntax error

Lỗi 7: gõ 'canceled' (sai chính tả so với dữ liệu):
   kết quả : [('2013-10-01', 0.0), ('2013-10-03', 0.0)]
   đúng là : [('2013-10-01', 0.33), ('2013-10-03', 0.67)] -> SAI

Lỗi 8: ROUND chỉ bọc tử số -> [('2013-10-01', 0.3333333333333333), ('2013-10-03', 0.6666666666666666)]


## 17. Mẹo nhận dạng pattern

| Dấu hiệu trong đề | Nghĩ ngay tới |
|---|---|
| "rate", "percentage", "ratio", "tỉ lệ … mỗi ngày / mỗi nhóm" | `GROUP BY` nhóm + `AVG(điều kiện)` hoặc `1.0 * SUM(CASE …) / COUNT(*)` |
| một bảng đóng **nhiều vai trò** (khách/tài xế, nhân viên/quản lý, người gửi/người nhận) | JOIN bảng đó **nhiều lần**, mỗi lần một alias |
| "loại bỏ những X thuộc nhóm Y" | `NOT IN (subquery)`, `NOT EXISTS`, hoặc JOIN rồi lọc ở `WHERE` |
| "between ngày A và ngày B" | `BETWEEN` (lấy cả hai đầu); kiểm tra định dạng ngày |
| "round to 2 decimal places" | `ROUND(…, 2)` bọc **ngoài** phép chia; kiểm tra phép chia có ra số thực không |
| "with at least one …" | thường tự đúng nhờ "lọc trước, gom nhóm sau"; nếu điều kiện đặt trên **nhóm** thì dùng `HAVING` |
| tên cột kết quả có dấu cách / chữ hoa | đặt alias đúng y hệt, có quote |

**Quy trình 4 bước cho mọi bài "tỉ lệ theo nhóm":** (1) xác định **hàng hợp lệ** (`FROM`/`JOIN`/`WHERE`); (2) xác định **khóa nhóm** (`GROUP BY`); (3) **tử số** = đếm có điều kiện, **mẫu số** = `COUNT(*)`; (4) **chia thực rồi làm tròn**.

## 18. Biến thể và bài liên quan

- **Tỉ lệ hủy theo thành phố** thay vì theo ngày: chỉ cần đổi `GROUP BY t.request_at` thành `GROUP BY t.city_id` (Bài tập 5).
- **Tách tỉ lệ hủy do tài xế và do khách:** hai biểu thức `AVG(status = 'cancelled_by_driver')` và `AVG(status = 'cancelled_by_client')` trong cùng một `SELECT` (Bài tập 6).
- **Chỉ lấy ngày có ít nhất `k` chuyến hợp lệ:** điều kiện đặt trên **nhóm**, không phải trên hàng, nên phải dùng **`HAVING`**. `HAVING` giống `WHERE` nhưng chạy **sau** `GROUP BY` và được dùng hàm gộp (`HAVING COUNT(*) >= 3`). Còn `WHERE` chạy trước khi gom nhóm nên **không** được dùng `COUNT(*)`. Xem ví dụ ở cell dưới.
- **Ngày có tỉ lệ hủy cao nhất:** thêm ``ORDER BY `Cancellation Rate` DESC LIMIT 1`` (nếu có nhiều ngày bằng nhau thì cần subquery hoặc window function).
- **Muốn in cả ngày không có chuyến nào với tỉ lệ 0:** cần một bảng "lịch" chứa mọi ngày, rồi `LEFT JOIN` từ bảng lịch sang (đề 262 **không** yêu cầu).

Bài LeetCode luyện cùng kỹ năng:

| Bài | Kỹ năng chung với #262 |
|---|---|
| #181 Employees Earning More Than Their Managers | JOIN một bảng với **chính nó** (self-join, alias) |
| #1211 Queries Quality and Percentage | `AVG(điều kiện)` theo nhóm + `ROUND` |
| #1633 Percentage of Users Attended a Contest | tỉ lệ theo nhóm, phép chia + `ROUND` |
| #1934 Confirmation Rate | `LEFT JOIN` + `AVG(điều kiện)`, xử lý nhóm không có dữ liệu |
| #1174 Immediate Food Delivery II | tỉ lệ phần trăm trên tập hàng đã lọc bằng subquery |
| #550 Game Play Analysis IV | tỉ lệ trên tập người dùng + subquery |
| #185 Department Top Three Salaries | bài Database Hard khác: JOIN + subquery / window function |

In [32]:
print("HAVING: chỉ giữ những ngày có ít nhất 3 chuyến hợp lệ (Example 1)")
show(conn_ex1, """
SELECT t.request_at AS `Day`,
       COUNT(*) AS valid_trips,
       ROUND(AVG(t.status != 'completed'), 2) AS `Cancellation Rate`
FROM Trips t
JOIN Users c ON t.client_id = c.users_id
JOIN Users d ON t.driver_id = d.users_id
WHERE c.banned = 'No' AND d.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at
HAVING COUNT(*) >= 3
""")

HAVING: chỉ giữ những ngày có ít nhất 3 chuyến hợp lệ (Example 1)
+------------+-------------+-------------------+
| Day        | valid_trips | Cancellation Rate |
+------------+-------------+-------------------+
| 2013-10-01 | 3           | 0.33              |
+------------+-------------+-------------------+ (1 hàng)


## 19. Bài tập vận dụng (tăng dần)

Làm theo thứ tự. Các cell TODO chỉ **định nghĩa hàm**, không tự chạy. Làm xong thì bỏ dấu `#` ở dòng test để tự kiểm. Gợi ý ở mục 20, đáp án ở mục 21 (cuối notebook).

### Bài 1: Đoán output (không chạy code)

Trên **SQLite**, mỗi biểu thức sau cho ra gì?

```sql
a) SELECT 7 / 2;
b) SELECT 7 / 2.0;
c) SELECT ROUND(2.0 / 3, 2);
d) SELECT 'completed' != 'completed', 'cancelled_by_driver' != 'completed';
e) SELECT '2013-10-10' BETWEEN '2013-10-01' AND '2013-10-03';
f) SELECT '2013-10-2'  BETWEEN '2013-10-01' AND '2013-10-03';
```

Câu hỏi thêm: `SELECT 7 / 2;` trên **MySQL** cho ra gì?

### Bài 2: Tính tay

`Users`: `(1, No, client)`, `(2, No, client)`, `(3, Yes, driver)`, `(4, No, driver)`.

| id | client_id | driver_id | city_id | status | request_at |
|---:|---:|---:|---:|---|---|
| 1 | 1 | 4 | 1 | completed | 2013-10-01 |
| 2 | 2 | 4 | 1 | cancelled_by_client | 2013-10-01 |
| 3 | 1 | 3 | 1 | cancelled_by_driver | 2013-10-01 |
| 4 | 2 | 3 | 2 | completed | 2013-10-02 |
| 5 | 1 | 4 | 2 | cancelled_by_driver | 2013-10-03 |
| 6 | 2 | 4 | 2 | cancelled_by_client | 2013-10-03 |
| 7 | 1 | 4 | 1 | completed | 2013-10-03 |
| 8 | 2 | 4 | 1 | cancelled_by_client | 2013-10-05 |

(a) Lập bảng kết quả đúng `(Day, Cancellation Rate)`. (b) Nếu **quên lọc tài xế** bị cấm thì kết quả là gì? (c) Nếu viết `SUM(...) / COUNT(*)` (không có `1.0`) trên SQLite thì kết quả là gì?

### Bài 3: Điền chỗ trống vào query JOIN hai lần

Hoàn thiện 8 chỗ `TODO` trong câu SQL của hàm `exercise_3_query()`, rồi xóa dòng `raise`.

In [33]:
def exercise_3_query() -> str:
    sql = """
    SELECT
        t.request_at AS `Day`,
        ROUND(1.0 * /* TODO 1: đếm số chuyến bị hủy */ / /* TODO 2: đếm tổng số chuyến */, 2)
            AS `Cancellation Rate`
    FROM Trips AS t
    JOIN Users AS c ON /* TODO 3: nối khách */
    JOIN Users AS d ON /* TODO 4: nối tài xế */
    WHERE /* TODO 5: khách không bị cấm */
      AND /* TODO 6: tài xế không bị cấm */
      AND /* TODO 7: khoảng ngày */
    GROUP BY /* TODO 8 */
    """
    raise NotImplementedError("Điền xong các TODO thì xóa dòng raise này")
    return sql

# Tự làm xong rồi mới mở test:
# check_day_query(exercise_3_query())

### Bài 4: Tự viết lại, ghép JOIN hai lần với `AVG`

Cách 4 trong bài dùng `AVG` với bộ lọc `NOT IN`. Hãy tự viết (không nhìn lại) một query dùng **JOIN `Users` hai lần** để lọc, và **`AVG`** để tính tỉ lệ. Không được dùng `SUM`, `COUNT` hay `1.0 *`.

In [34]:
def exercise_4_query() -> str:
    # TODO: trả về một chuỗi SQL: JOIN Users hai lần (alias c, d) + WHERE + GROUP BY
    #       và cột `Cancellation Rate` tính bằng ROUND(AVG(...), 2)
    raise NotImplementedError

# Gợi ý test:
# check_day_query(exercise_4_query())

### Bài 5: Biến thể, tỉ lệ hủy theo **thành phố**

Vẫn chỉ tính chuyến hợp lệ (cả hai người không bị cấm, ngày trong `[2013-10-01, 2013-10-03]`), nhưng gom theo `city_id`. Kết quả có 3 cột `city_id`, `total_trips` (số chuyến hợp lệ), `cancel_rate` (làm tròn 2 chữ số), **sắp xếp theo `city_id` tăng dần**.

Kết quả mong đợi trên bộ dữ liệu khó: `[(1, 1, 0.0), (6, 3, 0.33), (12, 2, 1.0)]`. Trên Example 1: `[(1, 2, 0.0), (6, 3, 0.33), (12, 2, 0.5)]`.

In [35]:
def exercise_5_query() -> str:
    # TODO: lọc như lời giải chính, nhưng GROUP BY t.city_id
    #       SELECT 3 cột: city_id, total_trips, cancel_rate; nhớ ORDER BY city_id
    raise NotImplementedError

# Gợi ý test:
# assert run_query(conn_hard, exercise_5_query())[1] == [(1, 1, 0.0), (6, 3, 0.33), (12, 2, 1.0)]
# assert run_query(conn_ex1, exercise_5_query())[1] == [(1, 2, 0.0), (6, 3, 0.33), (12, 2, 0.5)]

### Bài 6: Mini challenge, tách hai loại hủy + `HAVING`

Với mỗi ngày trong khoảng, chỉ tính chuyến hợp lệ, in 3 cột:

- `Day`
- `driver_rate`: tỉ lệ chuyến **tài xế hủy** (`cancelled_by_driver`), làm tròn 2 chữ số.
- `client_rate`: tỉ lệ chuyến **khách hủy** (`cancelled_by_client`), làm tròn 2 chữ số.

Chỉ giữ những ngày có **ít nhất 3 chuyến hợp lệ**, sắp xếp theo ngày tăng dần.

Mong đợi trên bộ khó: `[('2013-10-01', 0.33, 0.0), ('2013-10-03', 0.33, 0.33)]`. Trên Example 1: `[('2013-10-01', 0.0, 0.33)]`.

Câu hỏi suy nghĩ thêm: `driver_rate + client_rate` có luôn bằng `Cancellation Rate` của đề không? (Gợi ý: nghĩ về việc làm tròn.)

In [36]:
def exercise_6_query() -> str:
    # TODO: hai cột AVG(điều kiện) khác nhau, GROUP BY ngày, HAVING COUNT(*) >= 3, ORDER BY ngày
    raise NotImplementedError

# Gợi ý test:
# assert run_query(conn_hard, exercise_6_query())[1] == [("2013-10-01", 0.33, 0.0), ("2013-10-03", 0.33, 0.33)]
# assert run_query(conn_ex1, exercise_6_query())[1] == [("2013-10-01", 0.0, 0.33)]

## 20. Gợi ý theo mức

Đọc gợi ý mức 1 trước, bí quá mới đọc mức 2, rồi mức 3.

**Bài 1**
- Mức 1: nhớ lại bảng ở mục 8 (chia nguyên) và mục 4 (so sánh chuỗi từng ký tự).
- Mức 2: SQLite chia nguyên khi **cả hai** vế là số nguyên; phép so sánh trả 1/0; `BETWEEN` so chuỗi.
- Mức 3: với e) và f), so ký tự thứ 9 của hai chuỗi với `'0'` của `'2013-10-03'`.

**Bài 2**
- Mức 1: làm như bảng trace của Cách 1: đi từng chuyến, quyết định "tính" hay "bỏ".
- Mức 2: ai bị cấm? Chỉ tài xế số 3. Chuyến nào có `driver_id = 3`? Chuyến nào ngoài khoảng ngày?
- Mức 3: ngày `01` còn chuyến 1, 2; ngày `02` không còn chuyến nào; ngày `03` còn chuyến 5, 6, 7.

**Bài 3**
- Mức 1: nhìn lại 3 bước xây query ở mục 10.
- Mức 2: TODO 1 là một `SUM(CASE WHEN … THEN 1 ELSE 0 END)`; TODO 2 là `COUNT(*)`; TODO 3, 4 có dạng `t.xxx_id = c.users_id`.
- Mức 3: TODO 5–7 là `c.banned = 'No'`, `d.banned = 'No'`, `t.request_at BETWEEN '…' AND '…'`; TODO 8 là `t.request_at`.

**Bài 4**
- Mức 1: phần `FROM … WHERE … GROUP BY` giống hệt Bài 3.
- Mức 2: tỉ lệ = trung bình của chỉ báo 0/1.
- Mức 3: ``ROUND(AVG(t.status != 'completed'), 2) AS `Cancellation Rate` ``.

**Bài 5**
- Mức 1: chỉ đổi **khóa nhóm**.
- Mức 2: `SELECT t.city_id, COUNT(*) AS total_trips, ROUND(AVG(…), 2) AS cancel_rate`.
- Mức 3: `GROUP BY t.city_id ORDER BY t.city_id`.

**Bài 6**
- Mức 1: mỗi loại hủy là một chỉ báo 0/1 riêng.
- Mức 2: `AVG(t.status = 'cancelled_by_driver')` và `AVG(t.status = 'cancelled_by_client')`.
- Mức 3: thứ tự mệnh đề: `… WHERE … GROUP BY t.request_at HAVING COUNT(*) >= 3 ORDER BY t.request_at`.

## 21. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

### Đáp án Bài 1

| Câu | SQLite | Giải thích |
|---|---|---|
| a) `7 / 2` | `3` | hai số nguyên nên chia nguyên |
| b) `7 / 2.0` | `3.5` | có một vế số thực |
| c) `ROUND(2.0 / 3, 2)` | `0.67` | `0.6666…` làm tròn |
| d) | `0` và `1` | so sánh trả 0/1 |
| e) `'2013-10-10' BETWEEN …` | `0` | ký tự thứ 9: `'1' > '0'`, nên `'2013-10-10' > '2013-10-03'` |
| f) `'2013-10-2' BETWEEN …` | `0` | ký tự thứ 9: `'2' > '0'`, nên lớn hơn `'2013-10-03'`, dù "ngày 2" về ý nghĩa nằm trong khoảng! |

Trên **MySQL**, `7 / 2` cho `3.5000` vì `/` luôn chia thực.

### Đáp án Bài 2

(a) Chỉ tài xế 3 bị cấm, nên bỏ chuyến 3 và 4; chuyến 8 ngoài khoảng ngày. Ngày `01` còn chuyến 1, 2, có 1 hủy / 2, ra `0.50`. Ngày `02` không còn chuyến nào nên **không có hàng**. Ngày `03` còn chuyến 5, 6, 7, có 2 hủy / 3, ra `0.67`.

(b) Quên lọc tài xế: ngày `01` có 2 hủy / 3, ra `0.67`; ngày `02` **xuất hiện** với `0/1 = 0.00`; ngày `03` vẫn là `0.67`.

(c) Chia nguyên trên SQLite: `1/2 = 0` và `2/3 = 0`, nên cả hai ngày đều ra `0.0`.

In [37]:
print("Bài 1 (chạy thật trên SQLite):")
show(demo, """
SELECT 7 / 2 AS a, 7 / 2.0 AS b, ROUND(2.0 / 3, 2) AS c,
       'completed' != 'completed' AS d1, 'cancelled_by_driver' != 'completed' AS d2,
       '2013-10-10' BETWEEN '2013-10-01' AND '2013-10-03' AS e,
       '2013-10-2'  BETWEEN '2013-10-01' AND '2013-10-03' AS f
""")

USERS_B2 = [(1, "No", "client"), (2, "No", "client"), (3, "Yes", "driver"), (4, "No", "driver")]
TRIPS_B2 = [
    (1, 1, 4, 1, "completed", "2013-10-01"),
    (2, 2, 4, 1, "cancelled_by_client", "2013-10-01"),
    (3, 1, 3, 1, "cancelled_by_driver", "2013-10-01"),
    (4, 2, 3, 2, "completed", "2013-10-02"),
    (5, 1, 4, 2, "cancelled_by_driver", "2013-10-03"),
    (6, 2, 4, 2, "cancelled_by_client", "2013-10-03"),
    (7, 1, 4, 1, "completed", "2013-10-03"),
    (8, 2, 4, 1, "cancelled_by_client", "2013-10-05"),
]
conn_b2 = build_db(TRIPS_B2, USERS_B2)
answer_2a = normalize(run_query(conn_b2, SQL_JOIN)[1])
answer_2b = normalize(run_query(conn_b2, WRONG_NO_DRIVER)[1])
answer_2c = normalize(run_query(conn_b2, WRONG_INT_DIV)[1])
print("Bài 2a:", answer_2a)
print("Bài 2b:", answer_2b)
print("Bài 2c:", answer_2c)
assert answer_2a == [("2013-10-01", 0.5), ("2013-10-03", 0.67)]
assert answer_2b == [("2013-10-01", 0.67), ("2013-10-02", 0.0), ("2013-10-03", 0.67)]
assert answer_2c == [("2013-10-01", 0.0), ("2013-10-03", 0.0)]
assert answer_2a == via_python(TRIPS_B2, USERS_B2)

Bài 1 (chạy thật trên SQLite):
+---+-----+------+----+----+---+---+
| a | b   | c    | d1 | d2 | e | f |
+---+-----+------+----+----+---+---+
| 3 | 3.5 | 0.67 | 0  | 1  | 0 | 0 |
+---+-----+------+----+----+---+---+ (1 hàng)
Bài 2a: [('2013-10-01', 0.5), ('2013-10-03', 0.67)]
Bài 2b: [('2013-10-01', 0.67), ('2013-10-02', 0.0), ('2013-10-03', 0.67)]
Bài 2c: [('2013-10-01', 0.0), ('2013-10-03', 0.0)]


### Đáp án Bài 3 và Bài 4

Bài 3 chính là lời giải chính `SQL_JOIN`. Bài 4 giữ nguyên phần lọc của Bài 3 và thay công thức tỉ lệ bằng `AVG`.

In [38]:
def answer_3_query() -> str:
    return """
    SELECT
        t.request_at AS `Day`,
        ROUND(1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2)
            AS `Cancellation Rate`
    FROM Trips AS t
    JOIN Users AS c ON t.client_id = c.users_id
    JOIN Users AS d ON t.driver_id = d.users_id
    WHERE c.banned = 'No'
      AND d.banned = 'No'
      AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
    GROUP BY t.request_at
    """


def answer_4_query() -> str:
    return """
    SELECT
        t.request_at AS `Day`,
        ROUND(AVG(t.status != 'completed'), 2) AS `Cancellation Rate`
    FROM Trips AS t
    JOIN Users AS c ON t.client_id = c.users_id
    JOIN Users AS d ON t.driver_id = d.users_id
    WHERE c.banned = 'No'
      AND d.banned = 'No'
      AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
    GROUP BY t.request_at
    """


check_day_query(answer_3_query())
check_day_query(answer_4_query())

# Đối chiếu thêm với Cách 1 trên 100 bộ ngẫu nhiên
random.seed(2620)
for _ in range(100):
    trips_rows, users_rows = random_case()
    reference = via_python(trips_rows, users_rows)
    assert via_sql(answer_3_query())(trips_rows, users_rows) == reference
    assert via_sql(answer_4_query())(trips_rows, users_rows) == reference
print("Bài 3 và Bài 4 khớp Cách 1 trên 100 bộ ngẫu nhiên.")

PASS: Example 1 + bộ dữ liệu khó
PASS: Example 1 + bộ dữ liệu khó
Bài 3 và Bài 4 khớp Cách 1 trên 100 bộ ngẫu nhiên.


### Đáp án Bài 5 và Bài 6

Bài 5 chỉ đổi khóa nhóm thành `city_id`. Bài 6 dùng hai chỉ báo 0/1 khác nhau trong `AVG`, rồi lọc **nhóm** bằng `HAVING`.

Về câu hỏi thêm của Bài 6: **không phải lúc nào cũng bằng**. Về giá trị chính xác thì `driver_rate + client_rate` bằng tỉ lệ hủy, nhưng mỗi số được **làm tròn riêng** nên tổng có thể lệch `0.01`. Ví dụ ngày có 1 tài xế hủy, 1 khách hủy trên 3 chuyến: `0.33 + 0.33 = 0.66`, trong khi `ROUND(2/3, 2) = 0.67`. Ngày `03` của bộ khó chính là ca này.

In [39]:
def answer_5_query() -> str:
    return """
    SELECT t.city_id,
           COUNT(*) AS total_trips,
           ROUND(AVG(t.status != 'completed'), 2) AS cancel_rate
    FROM Trips AS t
    JOIN Users AS c ON t.client_id = c.users_id
    JOIN Users AS d ON t.driver_id = d.users_id
    WHERE c.banned = 'No'
      AND d.banned = 'No'
      AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
    GROUP BY t.city_id
    ORDER BY t.city_id
    """


def answer_6_query() -> str:
    return """
    SELECT t.request_at AS `Day`,
           ROUND(AVG(t.status = 'cancelled_by_driver'), 2) AS driver_rate,
           ROUND(AVG(t.status = 'cancelled_by_client'), 2) AS client_rate
    FROM Trips AS t
    JOIN Users AS c ON t.client_id = c.users_id
    JOIN Users AS d ON t.driver_id = d.users_id
    WHERE c.banned = 'No'
      AND d.banned = 'No'
      AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
    GROUP BY t.request_at
    HAVING COUNT(*) >= 3
    ORDER BY t.request_at
    """


show(conn_hard, answer_5_query())
assert run_query(conn_hard, answer_5_query())[1] == [(1, 1, 0.0), (6, 3, 0.33), (12, 2, 1.0)]
assert run_query(conn_ex1, answer_5_query())[1] == [(1, 2, 0.0), (6, 3, 0.33), (12, 2, 0.5)]

show(conn_hard, answer_6_query())
assert run_query(conn_hard, answer_6_query())[1] == [("2013-10-01", 0.33, 0.0), ("2013-10-03", 0.33, 0.33)]
assert run_query(conn_ex1, answer_6_query())[1] == [("2013-10-01", 0.0, 0.33)]
print("Đáp án Bài 5 và Bài 6 PASS.")

+---------+-------------+-------------+
| city_id | total_trips | cancel_rate |
+---------+-------------+-------------+
| 1       | 1           | 0.0         |
| 6       | 3           | 0.33        |
| 12      | 2           | 1.0         |
+---------+-------------+-------------+ (3 hàng)
+------------+-------------+-------------+
| Day        | driver_rate | client_rate |
+------------+-------------+-------------+
| 2013-10-01 | 0.33        | 0.0         |
| 2013-10-03 | 0.33        | 0.33        |
+------------+-------------+-------------+ (2 hàng)
Đáp án Bài 5 và Bài 6 PASS.


## 22. Template ngắn nên nhớ

**Công thức một dòng:** *lọc hàng hợp lệ (JOIN×2 hoặc NOT IN, cộng BETWEEN), rồi `GROUP BY` ngày, rồi `ROUND(1.0 * SUM(CASE …) / COUNT(*), 2)` hoặc `ROUND(AVG(điều kiện), 2)`.*

Bản chính (chạy ở mọi hệ quản trị):

```sql
SELECT
    t.request_at AS `Day`,
    ROUND(1.0 * SUM(CASE WHEN t.status != 'completed' THEN 1 ELSE 0 END) / COUNT(*), 2)
        AS `Cancellation Rate`
FROM Trips AS t
JOIN Users AS c ON t.client_id = c.users_id     -- khách
JOIN Users AS d ON t.driver_id = d.users_id     -- tài xế
WHERE c.banned = 'No'
  AND d.banned = 'No'
  AND t.request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY t.request_at;
```

Bản ngắn (MySQL / SQLite):

```sql
SELECT request_at AS `Day`, ROUND(AVG(status != 'completed'), 2) AS `Cancellation Rate`
FROM Trips
WHERE client_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND driver_id NOT IN (SELECT users_id FROM Users WHERE banned = 'Yes')
  AND request_at BETWEEN '2013-10-01' AND '2013-10-03'
GROUP BY request_at;
```

Bản Pandas:

```python
def trips_and_users(trips: pd.DataFrame, users: pd.DataFrame) -> pd.DataFrame:
    banned_ids = users.loc[users["banned"] == "Yes", "users_id"]
    valid = trips[(~trips["client_id"].isin(banned_ids)) & (~trips["driver_id"].isin(banned_ids))
                  & (trips["request_at"].between("2013-10-01", "2013-10-03"))]
    result = (valid.assign(is_cancelled=valid["status"] != "completed")
              .groupby("request_at", as_index=False)["is_cancelled"].mean())
    result["is_cancelled"] = result["is_cancelled"].round(2)
    return result.rename(columns={"request_at": "Day", "is_cancelled": "Cancellation Rate"})
```

## 23. Checklist trước khi submit

- [ ] Đã chọn đúng ngôn ngữ **MySQL** (hoặc Pandas) trên LeetCode chưa?
- [ ] Đã lọc **cả khách lẫn tài xế** bị cấm chưa? (Example 1 không bắt được lỗi này!)
- [ ] Có `BETWEEN '2013-10-01' AND '2013-10-03'` (lấy cả hai đầu mút) chưa?
- [ ] Phép chia ra **số thực** chưa? MySQL thì tự đúng; SQLite/PostgreSQL cần `1.0 *` đặt **trước** phép chia, hoặc dùng `AVG`.
- [ ] `ROUND(…, 2)` có bọc **toàn bộ** phép chia không?
- [ ] Tên cột đúng `Day` và `Cancellation Rate` (có quote vì có dấu cách) chưa?
- [ ] Dùng INNER JOIN, hoặc nếu LEFT JOIN thì điều kiện cấm nằm ở `WHERE` chứ không ở `ON`?
- [ ] Không JOIN `Users` một lần với `OR`?
- [ ] Trạng thái viết đúng `'cancelled_…'` (hai chữ `l`); an toàn nhất là `status != 'completed'`?
- [ ] Nếu dùng `NOT IN (subquery)`: subquery có thể trả `NULL` không?
- [ ] Đã thử với một bộ dữ liệu có **tài xế bị cấm**, **chuyến ngoài khoảng ngày** và **ngày toàn chuyến bị cấm** chưa?